> Advanced rule-based Orbit Wars agent with predictive simulation, economic optimization, and hybrid-ML-ready architecture.

---

# 1. Introduction

This notebook contains an advanced competitive Orbit Wars agent focused on:

- Economic snowball optimization
- Predictive orbital interception
- Threat-aware expansion
- Multi-source coordinated attacks
- Enemy interdiction
- Elimination strategies
- Fleet conservation

The architecture is modular and designed to support future ML systems such as:

- Shot validators
- PPO policies
- Learned target scoring
- Replay distillation

---

# 2. The Agent

```python
"""
Orbit Wars — Improved Agent v2
================================
Key improvements over the v1 baseline (750 → target 1500+):

1.  NPV-based target scoring  – early captures have compounding value because
    production ships fund further attacks. A planet captured 10 turns sooner
    is worth far more than linear math suggests.

2.  Enemy-fleet interdiction  – detect enemy fleets in transit toward neutral
    planets and race there first, or arrive just after to pick up the pieces.

3.  Speed-bracket exploitation – fleet_speed is log-concave, so there are
    discrete ship-count thresholds where adding a few ships drops travel time
    by a full turn. The send-sizing logic now seeks these thresholds explicitly.

4.  Threat-map reserves       – each planet's reserve is now computed from a
    real threat-weighted sum of incoming enemy fleets, not a fixed fraction.

5.  Elimination surge mode    – when any enemy is below a strength threshold,
    ALL available ships are coordinated to eliminate them immediately.

6.  Production-avalanche mode – when our production exceeds the enemy's by a
    large margin, shift to aggressive expansion (85% attack budget).

7.  Better 4-player dynamics  – opponent priority uses strength * proximity
    product; we target the weakest nearby opponent to eliminate them fast.

8.  Improved opening rush     – a dedicated opening-turn prioritiser scores
    neutrals by (production / turns_to_reach)^1.5 and commits multiple
    planets simultaneously to grab a dominant early economy.
"""

import math
import time
from collections import defaultdict, namedtuple
from dataclasses import dataclass, field

# ============================================================
# Configuration
# ============================================================

BOARD       = 100.0
CENTER_X    = 50.0
CENTER_Y    = 50.0
SUN_R       = 10.0
MAX_SPEED   = 6.0
SUN_SAFETY  = 1.5
ROTATION_LIMIT = 50.0
TOTAL_STEPS    = 500
SIM_HORIZON    = 110
ROUTE_SEARCH_HORIZON = 60
HORIZON        = SIM_HORIZON
LAUNCH_CLEARANCE = 0.1

EARLY_TURN_LIMIT   = 65.000000
OPENING_TURN_LIMIT = 138.000000
LATE_REMAINING_TURNS      = 60
VERY_LATE_REMAINING_TURNS = 25

SAFE_NEUTRAL_MARGIN     = 2
CONTESTED_NEUTRAL_MARGIN = 2
INTERCEPT_TOLERANCE     = 1

SAFE_OPENING_PROD_THRESHOLD   = 4
SAFE_OPENING_TURN_LIMIT       = 10
ROTATING_OPENING_MAX_TURNS    = 13
ROTATING_OPENING_LOW_PROD     = 2
FOUR_PLAYER_ROTATING_REACTION_GAP  = 3
FOUR_PLAYER_ROTATING_SEND_RATIO   = 0.419945
FOUR_PLAYER_ROTATING_TURN_LIMIT   = 10

COMET_MAX_CHASE_TURNS = 10

# ── Cost weights (lower = more willing to pay ships/turns) ──
ATTACK_COST_TURN_WEIGHT  = 0.949673# was 0.55 — slightly more turn-tolerant
SNIPE_COST_TURN_WEIGHT   = 0.659442# was 0.45
INDIRECT_VALUE_SCALE     = 0.456832# was 0.15 — neighbourhood matters more
INDIRECT_FRIENDLY_WEIGHT = 0.35
INDIRECT_NEUTRAL_WEIGHT  = 0.90
INDIRECT_ENEMY_WEIGHT    = 1.30   # was 1.25 — nearby enemies cost us more

# ── Value multipliers ──
STATIC_NEUTRAL_VALUE_MULT         = 1.030055    # was 1.4
STATIC_HOSTILE_VALUE_MULT         = 1.275211   # was 1.55
ROTATING_OPENING_VALUE_MULT       = 1.580249
HOSTILE_TARGET_VALUE_MULT         = 2.115689# was 1.85 — be more aggressive
OPENING_HOSTILE_TARGET_VALUE_MULT = 1.55   # was 1.45
SAFE_NEUTRAL_VALUE_MULT           = 0.855319# was 1.2
CONTESTED_NEUTRAL_VALUE_MULT      = 0.913740
EARLY_NEUTRAL_VALUE_MULT          = 1.30   # was 1.2 — rush opening harder
COMET_VALUE_MULT                  = 0.65
SNIPE_VALUE_MULT                  = 1.15   # was 1.12
SWARM_VALUE_MULT                  = 1.06
REINFORCE_VALUE_MULT              = 1.35
CRASH_EXPLOIT_VALUE_MULT          = 1.20   # was 1.18
FINISHING_HOSTILE_VALUE_MULT      = 1.20   # was 1.15
BEHIND_ROTATING_NEUTRAL_VALUE_MULT = 0.92
INTERDICTION_VALUE_MULT           = 1.935345# NEW — arriving before enemy is gold
ELIMINATION_VALUE_MULT            = 1.768789# NEW — elimination is highest priority

# ── NPV / tempo bonuses (NEW) ──
NPV_COMPOUNDING_RATE     = 0.177508# bonus per early turn, capped
NPV_COMPOUNDING_CAP_TURN = 114.000000     # cap applies beyond this many turns
DENIAL_BONUS_MULT        = 0.393753# added to value when we're denying enemy a planet
EARLY_RUSH_TURNS         = 50     # within this, apply extra rush multiplier
EARLY_RUSH_MULT          = 0.499974   # per-turn bonus scaling * production

# ── Margins ──
NEUTRAL_MARGIN_BASE        = 1.000000
NEUTRAL_MARGIN_PROD_WEIGHT = 2
NEUTRAL_MARGIN_CAP         = 8
HOSTILE_MARGIN_BASE        = 11.000000
HOSTILE_MARGIN_PROD_WEIGHT = 2
HOSTILE_MARGIN_CAP         = 12
HOSTILE_REINFORCE_HORIZON  = 8
HOSTILE_REINFORCE_RATIO    = 0.336122
HOSTILE_REINFORCE_CAP      = 15
STATIC_TARGET_MARGIN       = 4
CONTESTED_TARGET_MARGIN    = 5
FOUR_PLAYER_TARGET_MARGIN  = 3
LONG_TRAVEL_MARGIN_START   = 18
LONG_TRAVEL_MARGIN_DIVISOR = 3
LONG_TRAVEL_MARGIN_CAP     = 8
COMET_MARGIN_RELIEF        = 6
FINISHING_HOSTILE_SEND_BONUS = 5

STATIC_TARGET_SCORE_MULT             = 1.810854   # was 1.18
EARLY_STATIC_NEUTRAL_SCORE_MULT      = 1.30   # was 1.25
FOUR_PLAYER_ROTATING_NEUTRAL_SCORE_MULT = 0.84
DENSE_STATIC_NEUTRAL_COUNT           = 4
DENSE_ROTATING_NEUTRAL_SCORE_MULT    = 0.86
SNIPE_SCORE_MULT                     = 1.725044   # was 1.12
SWARM_SCORE_MULT                     = 1.07
CRASH_EXPLOIT_SCORE_MULT             = 1.06

FOLLOWUP_MIN_SHIPS        = 8
LOW_VALUE_COMET_PRODUCTION = 1
LATE_CAPTURE_BUFFER       = 5
VERY_LATE_CAPTURE_BUFFER  = 3

DEFENSE_LOOKAHEAD_TURNS        = 43
DEFENSE_COST_TURN_WEIGHT       = 0.38   # was 0.4
DEFENSE_FRONTIER_SCORE_MULT    = 1.15   # was 1.12
DEFENSE_SEND_MARGIN_BASE       = 1
DEFENSE_SEND_MARGIN_PROD_WEIGHT = 1
DEFENSE_SHIP_VALUE             = 0.55

REINFORCE_ENABLED             = True
REINFORCE_MIN_PRODUCTION      = 2
REINFORCE_MAX_TRAVEL_TURNS    = 22
REINFORCE_SAFETY_MARGIN       = 2
REINFORCE_MAX_SOURCE_FRACTION = 0.75
REINFORCE_MIN_FUTURE_TURNS    = 40
REINFORCE_HOLD_LOOKAHEAD      = 20
REINFORCE_COST_TURN_WEIGHT    = 0.35

RECAPTURE_LOOKAHEAD_TURNS     = 10
RECAPTURE_COST_TURN_WEIGHT    = 0.52
RECAPTURE_VALUE_MULT          = 0.88
RECAPTURE_FRONTIER_MULT       = 1.08
RECAPTURE_PRODUCTION_WEIGHT   = 0.6
RECAPTURE_IMMEDIATE_WEIGHT    = 0.4

REAR_SOURCE_MIN_SHIPS       = 16
REAR_DISTANCE_RATIO         = 1.25
REAR_STAGE_PROGRESS         = 0.78
REAR_SEND_RATIO_TWO_PLAYER  = 0.426833   # was 0.62 — push more from rear
REAR_SEND_RATIO_FOUR_PLAYER = 0.414997   # was 0.7
REAR_SEND_MIN_SHIPS         = 10
REAR_MAX_TRAVEL_TURNS       = 40

PARTIAL_SOURCE_MIN_SHIPS     = 6
MULTI_SOURCE_TOP_K           = 5
MULTI_SOURCE_ETA_TOLERANCE   = 2
MULTI_SOURCE_PLAN_PENALTY    = 0.97
HOSTILE_SWARM_ETA_TOLERANCE  = 1
THREE_SOURCE_SWARM_ENABLED   = True
THREE_SOURCE_MIN_TARGET_SHIPS = 20
THREE_SOURCE_ETA_TOLERANCE   = 1
THREE_SOURCE_PLAN_PENALTY    = 0.93

PROACTIVE_DEFENSE_HORIZON       = 7.000000   # was 12 — look further ahead
PROACTIVE_DEFENSE_RATIO         = 0.649773  # was 0.18
MULTI_ENEMY_PROACTIVE_HORIZON   = 16   # was 14
MULTI_ENEMY_PROACTIVE_RATIO     = 0.24  # was 0.22
MULTI_ENEMY_STACK_WINDOW        = 3
REACTION_SOURCE_TOP_K_MY        = 4
REACTION_SOURCE_TOP_K_ENEMY     = 4
PROACTIVE_ENEMY_TOP_K           = 3

CRASH_EXPLOIT_ENABLED          = True
CRASH_EXPLOIT_MIN_TOTAL_SHIPS  = 10
CRASH_EXPLOIT_ETA_WINDOW       = 2
CRASH_EXPLOIT_POST_CRASH_DELAY = 1
SHADOW_SNIPE_INTENT_THRESHOLD  = 3.0   # was 3.5 — more trigger-happy

LATE_IMMEDIATE_SHIP_VALUE  = 1.789134   # was 0.6
WEAK_ENEMY_THRESHOLD       = 129.000000
ELIMINATION_BONUS          = 61.619694   # was 18.0 — bigger elimination reward
ELIMINATION_STRENGTH_RATIO = 0.218376# NEW — trigger when enemy < this fraction of us

BEHIND_DOMINATION    = -0.699607  # was -0.20 — recognise being behind sooner
AHEAD_DOMINATION     = 0.366624   # was 0.18
FINISHING_DOMINATION = 0.33
FINISHING_PROD_RATIO = 1.253682   # was 1.25 — enter finishing mode sooner
AHEAD_ATTACK_MARGIN_BONUS    = 0.14  # was 0.12
BEHIND_ATTACK_MARGIN_PENALTY = 0.03  # was 0.04
FINISHING_ATTACK_MARGIN_BONUS = 0.14 # was 0.12

# NEW — production avalanche mode
AVALANCHE_PROD_RATIO         = 2.343873# trigger if my_prod / enemy_prod > this
AVALANCHE_ATTACK_FRACTION    = 0.8574932200914216# attack budget when avalanche active

DOOMED_EVAC_TURN_LIMIT = 24
DOOMED_MIN_SHIPS       = 8

# NEW — interdiction parameters
INTERDICTION_HORIZON         = 35    # scan enemy fleets up to this many turns away
INTERDICTION_LEAD_TURNS      = 2     # arrive this many turns before the enemy
INTERDICTION_MIN_FLEET_SHIPS = 8     # only bother if enemy fleet is this big
INTERDICTION_MIN_PROD        = 2     # only interdict planets with at least this production

# NEW — elimination surge
ELIMINATION_SURGE_SHIPS      = 12    # commit this many ships minimum per surge planet
ELIMINATION_SURGE_TOP_K      = 3     # attack top-K enemy planets in surge mode

# NEW — speed bracket thresholds (ships at which travel time drops by 1 turn at dist=40)
# Pre-computed: [2,3,4,5,6,7,8,10,11,13,15,18,23,28,36,47,64,91,137,219,385]
SPEED_BRACKET_THRESHOLDS = [2,3,4,5,6,7,8,10,11,13,15,18,23,28,36,47,64,91,137,219,385,1000]

SOFT_ACT_DEADLINE        = 0.82
HEAVY_PHASE_MIN_TIME     = 0.16
OPTIONAL_PHASE_MIN_TIME  = 0.08
HEAVY_ROUTE_PLANET_LIMIT = 32


# ============================================================
# Shared Types
# ============================================================

Planet = namedtuple(
    "Planet", ["id", "owner", "x", "y", "radius", "ships", "production"]
)
Fleet = namedtuple(
    "Fleet", ["id", "owner", "x", "y", "angle", "from_planet_id", "ships"]
)


@dataclass(frozen=True)
class ShotOption:
    score:    float
    src_id:   int
    target_id: int
    angle:    float
    turns:    int
    needed:   int
    send_cap: int
    mission:  str          = "capture"
    anchor_turn: int | None = None


@dataclass
class Mission:
    kind:      str
    score:     float
    target_id: int
    turns:     int
    options:   list[ShotOption] = field(default_factory=list)


# ============================================================
# Physics
# ============================================================

def dist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)


def orbital_radius(planet):
    return dist(planet.x, planet.y, CENTER_X, CENTER_Y)


def is_static_planet(planet):
    return orbital_radius(planet) + planet.radius >= ROTATION_LIMIT


def fleet_speed(ships):
    if ships <= 1:
        return 1.0
    ratio = math.log(ships) / math.log(1000.0)
    ratio = max(0.0, min(1.0, ratio))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)


def point_to_segment_distance(px, py, x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    seg_len_sq = dx * dx + dy * dy
    if seg_len_sq <= 1e-9:
        return dist(px, py, x1, y1)
    t = ((px - x1) * dx + (py - y1) * dy) / seg_len_sq
    t = max(0.0, min(1.0, t))
    return dist(px, py, x1 + t * dx, y1 + t * dy)


def segment_hits_sun(x1, y1, x2, y2, safety=SUN_SAFETY):
    return (
        point_to_segment_distance(CENTER_X, CENTER_Y, x1, y1, x2, y2)
        < SUN_R + safety
    )


def launch_point(sx, sy, sr, angle):
    c = sr + LAUNCH_CLEARANCE
    return sx + math.cos(angle) * c, sy + math.sin(angle) * c


def actual_path_geometry(sx, sy, sr, tx, ty, tr):
    angle = math.atan2(ty - sy, tx - sx)
    start_x, start_y = launch_point(sx, sy, sr, angle)
    hit_distance = max(0.0, dist(sx, sy, tx, ty) - (sr + LAUNCH_CLEARANCE) - tr)
    end_x = start_x + math.cos(angle) * hit_distance
    end_y = start_y + math.sin(angle) * hit_distance
    return angle, start_x, start_y, end_x, end_y, hit_distance


def safe_angle_and_distance(sx, sy, sr, tx, ty, tr):
    angle, start_x, start_y, end_x, end_y, hit_distance = actual_path_geometry(
        sx, sy, sr, tx, ty, tr
    )
    if segment_hits_sun(start_x, start_y, end_x, end_y):
        return None
    return angle, hit_distance


def predict_planet_position(planet, initial_by_id, angular_velocity, turns):
    init = initial_by_id.get(planet.id)
    if init is None:
        return planet.x, planet.y
    r = dist(init.x, init.y, CENTER_X, CENTER_Y)
    if r + init.radius >= ROTATION_LIMIT:
        return planet.x, planet.y
    cur_ang = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    new_ang  = cur_ang + angular_velocity * turns
    return CENTER_X + r * math.cos(new_ang), CENTER_Y + r * math.sin(new_ang)


def predict_comet_position(planet_id, comets, turns):
    for group in comets:
        pids = group.get("planet_ids", [])
        if planet_id not in pids:
            continue
        idx   = pids.index(planet_id)
        paths = group.get("paths", [])
        path_index = group.get("path_index", 0)
        if idx >= len(paths):
            return None
        path = paths[idx]
        future_idx = path_index + int(turns)
        if 0 <= future_idx < len(path):
            return path[future_idx][0], path[future_idx][1]
        return None
    return None


def comet_remaining_life(planet_id, comets):
    for group in comets:
        pids = group.get("planet_ids", [])
        if planet_id not in pids:
            continue
        idx  = pids.index(planet_id)
        paths = group.get("paths", [])
        path_index = group.get("path_index", 0)
        if idx < len(paths):
            return max(0, len(paths[idx]) - path_index)
    return 0


def estimate_arrival(sx, sy, sr, tx, ty, tr, ships):
    safe = safe_angle_and_distance(sx, sy, sr, tx, ty, tr)
    if safe is None:
        return None
    angle, total_d = safe
    turns = max(1, int(math.ceil(total_d / fleet_speed(max(1, ships)))))
    return angle, turns


def travel_time(sx, sy, sr, tx, ty, tr, ships):
    est = estimate_arrival(sx, sy, sr, tx, ty, tr, ships)
    if est is None:
        return 10 ** 9
    return est[1]


def predict_target_position(target, turns, initial_by_id, ang_vel, comets, comet_ids):
    if target.id in comet_ids:
        return predict_comet_position(target.id, comets, turns)
    return predict_planet_position(target, initial_by_id, ang_vel, turns)


def target_can_move(target, initial_by_id, comet_ids):
    if target.id in comet_ids:
        return True
    init = initial_by_id.get(target.id)
    if init is None:
        return False
    r = dist(init.x, init.y, CENTER_X, CENTER_Y)
    return r + init.radius < ROTATION_LIMIT


def search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids):
    best       = None
    best_score = None
    max_turns  = min(HORIZON, ROUTE_SEARCH_HORIZON)
    if target.id in comet_ids:
        max_turns = min(max_turns, max(0, comet_remaining_life(target.id, comets) - 1))

    for candidate_turns in range(1, max_turns + 1):
        pos = predict_target_position(target, candidate_turns, initial_by_id, ang_vel, comets, comet_ids)
        if pos is None:
            continue
        est = estimate_arrival(src.x, src.y, src.radius, pos[0], pos[1], target.radius, ships)
        if est is None:
            continue
        _, turns = est
        if abs(turns - candidate_turns) > INTERCEPT_TOLERANCE:
            continue

        actual_turns = max(turns, candidate_turns)
        actual_pos   = predict_target_position(target, actual_turns, initial_by_id, ang_vel, comets, comet_ids)
        if actual_pos is None:
            continue

        confirm = estimate_arrival(src.x, src.y, src.radius, actual_pos[0], actual_pos[1], target.radius, ships)
        if confirm is None:
            continue

        delta = abs(confirm[1] - actual_turns)
        if delta > INTERCEPT_TOLERANCE:
            continue

        score = (delta, confirm[1], candidate_turns)
        if best is None or score < best_score:
            best_score = score
            best = (confirm[0], confirm[1], actual_pos[0], actual_pos[1])

    return best


def aim_with_prediction(src, target, ships, initial_by_id, ang_vel, comets, comet_ids):
    est = estimate_arrival(src.x, src.y, src.radius, target.x, target.y, target.radius, ships)
    if est is None:
        if not target_can_move(target, initial_by_id, comet_ids):
            return None
        return search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids)

    tx, ty = target.x, target.y
    for _ in range(5):
        _, turns = est
        pos = predict_target_position(target, turns, initial_by_id, ang_vel, comets, comet_ids)
        if pos is None:
            return None
        ntx, nty = pos
        next_est  = estimate_arrival(src.x, src.y, src.radius, ntx, nty, target.radius, ships)
        if next_est is None:
            if not target_can_move(target, initial_by_id, comet_ids):
                return None
            return search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids)
        if (
            abs(ntx - tx) < 0.3
            and abs(nty - ty) < 0.3
            and abs(next_est[1] - turns) <= INTERCEPT_TOLERANCE
        ):
            return next_est[0], next_est[1], ntx, nty
        tx, ty = ntx, nty
        est = next_est

    final_est = estimate_arrival(src.x, src.y, src.radius, tx, ty, target.radius, ships)
    if final_est is None:
        return search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids)
    return final_est[0], final_est[1], tx, ty


# ============================================================
# World Model helpers
# ============================================================

def fleet_target_planet(fleet, planets):
    best_planet = None
    best_time   = 1e9
    dir_x = math.cos(fleet.angle)
    dir_y = math.sin(fleet.angle)
    speed = fleet_speed(fleet.ships)

    for planet in planets:
        dx   = planet.x - fleet.x
        dy   = planet.y - fleet.y
        proj = dx * dir_x + dy * dir_y
        if proj < 0:
            continue
        perp_sq = dx * dx + dy * dy - proj * proj
        r_sq    = planet.radius * planet.radius
        if perp_sq >= r_sq:
            continue
        hit_d = max(0.0, proj - math.sqrt(max(0.0, r_sq - perp_sq)))
        turns = hit_d / speed
        if turns <= HORIZON and turns < best_time:
            best_time   = turns
            best_planet = planet

    if best_planet is None:
        return None, None
    return best_planet, int(math.ceil(best_time))


def build_arrival_ledger(fleets, planets):
    arrivals = {p.id: [] for p in planets}
    for fleet in fleets:
        target, eta = fleet_target_planet(fleet, planets)
        if target is None:
            continue
        arrivals[target.id].append((eta, fleet.owner, int(fleet.ships)))
    return arrivals


def resolve_arrival_event(owner, garrison, arrivals):
    by_owner = {}
    for _, attacker_owner, ships in arrivals:
        by_owner[attacker_owner] = by_owner.get(attacker_owner, 0) + ships

    if not by_owner:
        return owner, max(0.0, garrison)

    sorted_players = sorted(by_owner.items(), key=lambda item: item[1], reverse=True)
    top_owner, top_ships = sorted_players[0]

    if len(sorted_players) > 1:
        second_ships = sorted_players[1][1]
        if top_ships == second_ships:
            return -1, 0.0
        survivor_owner = top_owner
        survivor_ships = top_ships - second_ships
    else:
        survivor_owner = top_owner
        survivor_ships = top_ships

    if survivor_ships <= 0:
        return owner, max(0.0, garrison)
    if owner == survivor_owner:
        return owner, garrison + survivor_ships
    garrison -= survivor_ships
    if garrison < 0:
        return survivor_owner, -garrison
    return owner, garrison


def normalize_arrivals(arrivals, horizon):
    events = []
    for turns, owner, ships in arrivals:
        if ships <= 0:
            continue
        eta = max(1, int(math.ceil(turns)))
        if eta > horizon:
            continue
        events.append((eta, owner, int(ships)))
    events.sort(key=lambda item: item[0])
    return events


def simulate_planet_timeline(planet, arrivals, player, horizon):
    horizon = max(0, int(math.ceil(horizon)))
    events  = normalize_arrivals(arrivals, horizon)
    by_turn = defaultdict(list)
    for item in events:
        by_turn[item[0]].append(item)

    owner    = planet.owner
    garrison = float(planet.ships)
    owner_at = {0: owner}
    ships_at = {0: max(0.0, garrison)}
    min_owned   = garrison if owner == player else 0.0
    first_enemy = None
    fall_turn   = None

    for turn in range(1, horizon + 1):
        if owner != -1:
            garrison += planet.production

        group      = by_turn.get(turn, [])
        prev_owner = owner
        if group:
            if prev_owner == player and first_enemy is None:
                if any(item[1] not in (-1, player) for item in group):
                    first_enemy = turn
            owner, garrison = resolve_arrival_event(owner, garrison, group)
            if prev_owner == player and owner != player and fall_turn is None:
                fall_turn = turn

        owner_at[turn] = owner
        ships_at[turn] = max(0.0, garrison)
        if owner == player:
            min_owned = min(min_owned, garrison)

    keep_needed = 0
    holds_full  = True

    if planet.owner == player:
        def survives_with_keep(keep):
            sim_owner    = planet.owner
            sim_garrison = float(keep)
            for turn in range(1, horizon + 1):
                if sim_owner != -1:
                    sim_garrison += planet.production
                group = by_turn.get(turn, [])
                if group:
                    sim_owner, sim_garrison = resolve_arrival_event(sim_owner, sim_garrison, group)
                    if sim_owner != player:
                        return False
            return sim_owner == player

        if survives_with_keep(int(planet.ships)):
            lo, hi = 0, int(planet.ships)
            while lo < hi:
                mid = (lo + hi) // 2
                if survives_with_keep(mid):
                    hi = mid
                else:
                    lo = mid + 1
            keep_needed = lo
        else:
            holds_full  = False
            keep_needed = int(planet.ships)

    return {
        "owner_at":    owner_at,
        "ships_at":    ships_at,
        "keep_needed": keep_needed,
        "min_owned":   max(0, int(math.floor(min_owned))) if planet.owner == player else 0,
        "first_enemy": first_enemy,
        "fall_turn":   fall_turn,
        "holds_full":  holds_full,
        "horizon":     horizon,
    }


def state_at_timeline(timeline, arrival_turn):
    turn  = max(0, min(int(math.ceil(arrival_turn)), timeline["horizon"]))
    owner = timeline["owner_at"].get(turn, timeline["owner_at"][timeline["horizon"]])
    ships = timeline["ships_at"].get(turn, timeline["ships_at"][timeline["horizon"]])
    return owner, max(0.0, ships)


def count_players(planets, fleets):
    owners = set()
    for planet in planets:
        if planet.owner != -1:
            owners.add(planet.owner)
    for fleet in fleets:
        owners.add(fleet.owner)
    return max(2, len(owners))


def nearest_distance_to_set(px, py, planets):
    if not planets:
        return 10 ** 9
    return min(dist(px, py, p.x, p.y) for p in planets)


def indirect_features(planet, planets, player):
    friendly = neutral = enemy = 0.0
    for other in planets:
        if other.id == planet.id:
            continue
        d = dist(planet.x, planet.y, other.x, other.y)
        if d < 1:
            continue
        factor = other.production / (d + 12.0)
        if other.owner == player:
            friendly += factor
        elif other.owner == -1:
            neutral += factor
        else:
            enemy += factor
    return friendly, neutral, enemy


# ============================================================
# WorldModel
# ============================================================

class WorldModel:
    def __init__(self, player, step, planets, fleets, initial_by_id, ang_vel, comets, comet_ids):
        self.player        = player
        self.step          = step
        self.planets       = planets
        self.fleets        = fleets
        self.initial_by_id = initial_by_id
        self.ang_vel       = ang_vel
        self.comets        = comets
        self.comet_ids     = set(comet_ids)

        self.planet_by_id      = {p.id: p for p in planets}
        self.my_planets        = [p for p in planets if p.owner == player]
        self.enemy_planets     = [p for p in planets if p.owner not in (-1, player)]
        self.neutral_planets   = [p for p in planets if p.owner == -1]
        self.static_neutral_planets = [p for p in self.neutral_planets if is_static_planet(p)]

        self.num_players    = count_players(planets, fleets)
        self.remaining_steps = max(1, TOTAL_STEPS - step)
        self.is_early   = step < EARLY_TURN_LIMIT
        self.is_opening = step < OPENING_TURN_LIMIT
        self.is_late    = self.remaining_steps < LATE_REMAINING_TURNS
        self.is_very_late = self.remaining_steps < VERY_LATE_REMAINING_TURNS
        self.is_four_player = self.num_players >= 4

        self.owner_strength  = defaultdict(int)
        self.owner_production = defaultdict(int)
        for p in planets:
            if p.owner != -1:
                self.owner_strength[p.owner]   += int(p.ships)
                self.owner_production[p.owner] += int(p.production)
        for f in fleets:
            self.owner_strength[f.owner] += int(f.ships)

        self.my_total = self.owner_strength.get(player, 0)
        self.enemy_total = sum(
            v for k, v in self.owner_strength.items() if k != player
        )
        self.max_enemy_strength = max(
            (v for k, v in self.owner_strength.items() if k != player),
            default=0,
        )
        self.my_prod    = self.owner_production.get(player, 0)
        self.enemy_prod = sum(v for k, v in self.owner_production.items() if k != player)

        self.arrivals_by_planet = build_arrival_ledger(fleets, planets)
        self.base_timeline = {
            p.id: simulate_planet_timeline(
                p, self.arrivals_by_planet[p.id], player, HORIZON
            )
            for p in planets
        }
        self.keep_needed_map = {p.id: self.base_timeline[p.id]["keep_needed"] for p in planets}
        self.min_owned_map   = {p.id: self.base_timeline[p.id]["min_owned"]   for p in planets}
        self.first_enemy_map = {p.id: self.base_timeline[p.id]["first_enemy"] for p in planets}
        self.fall_turn_map   = {p.id: self.base_timeline[p.id]["fall_turn"]   for p in planets}
        self.holds_full_map  = {p.id: self.base_timeline[p.id]["holds_full"]  for p in planets}
        self.indirect_feature_map = {
            p.id: indirect_features(p, planets, player) for p in planets
        }

        self.shot_cache            = {}
        self.probe_candidate_cache = {}
        self.best_probe_cache      = {}
        self.reaction_cache        = {}
        self.exact_need_cache      = {}

        self.total_visible_ships = sum(int(p.ships) for p in planets) + sum(int(f.ships) for f in fleets)
        self.total_production    = sum(int(p.production) for p in planets)

        # Per-opponent proximity analysis
        self.opp_planets  = defaultdict(list)
        for p in self.enemy_planets:
            self.opp_planets[p.owner].append(p)
        self.opp_min_dist = {}
        for opp_id, opp_ps in self.opp_planets.items():
            self.opp_min_dist[opp_id] = min(
                dist(mp.x, mp.y, ep.x, ep.y)
                for mp in self.my_planets
                for ep in opp_ps
            ) if self.my_planets else 999

        self.enemy_intent_cache = {}

        # ── NEW: build threat map ────────────────────────────────────────────
        # threat_map[planet_id] = estimated enemy force that could arrive within
        # PROACTIVE_DEFENSE_HORIZON turns, weighted by proximity.
        self.threat_map = self._build_threat_map()

        # ── NEW: build enemy transit info ────────────────────────────────────
        # enemy_transits[target_planet_id] = list of (eta, owner, ships) for
        # enemy fleets that are currently heading to neutral or enemy planets.
        self.enemy_transits = self._build_enemy_transits()

    # ── threat map ──────────────────────────────────────────────────────────

    def _build_threat_map(self):
        threat = {p.id: 0.0 for p in self.my_planets}
        for p in self.my_planets:
            # In-flight enemy fleets
            for eta, owner, ships in self.arrivals_by_planet.get(p.id, []):
                if owner not in (-1, self.player) and eta > 0:
                    threat[p.id] += ships / max(1, eta)
            # Enemy planets within PROACTIVE_DEFENSE_HORIZON turns
            for ep in self.enemy_planets:
                aim = self.plan_shot(ep.id, p.id, max(1, int(ep.ships)))
                if aim is None:
                    continue
                eta = aim[1]
                if eta <= PROACTIVE_DEFENSE_HORIZON:
                    threat[p.id] += int(ep.ships) / max(1, eta)
        return threat

    # ── enemy transits ───────────────────────────────────────────────────────

    def _build_enemy_transits(self):
        transits = defaultdict(list)
        for fleet in self.fleets:
            if fleet.owner in (-1, self.player):
                continue
            target_p, eta = fleet_target_planet(fleet, self.planets)
            if target_p is None:
                continue
            if target_p.owner == fleet.owner:
                continue   # reinforcing their own — not interesting to us
            transits[target_p.id].append((eta, fleet.owner, int(fleet.ships)))
        return transits

    # ── unchanged helpers ────────────────────────────────────────────────────

    def predict_enemy_intent(self, opp_id):
        if opp_id in self.enemy_intent_cache:
            return self.enemy_intent_cache[opp_id]
        opp_ps = self.opp_planets.get(opp_id, [])
        if not opp_ps:
            return {}
        predictions = {}
        for target in self.planets:
            if target.owner == opp_id:
                continue
            best_score = 0
            for src in opp_ps:
                d = dist(src.x, src.y, target.x, target.y)
                if d < 5:
                    continue
                score = (target.production * 15.0 + 10.0) / (d + 10.0)
                if target.owner == -1:
                    score *= 1.3
                if target.owner == self.player:
                    score *= 0.8
                best_score = max(best_score, score)
            if best_score > 1.5:
                predictions[target.id] = best_score
        self.enemy_intent_cache[opp_id] = predictions
        return predictions

    def opponent_priority(self, opp_id):
        """Higher = attack this opponent first.  Weak nearby opponents score highest."""
        if not self.opp_min_dist:
            return 1.0
        my_dist   = self.opp_min_dist.get(opp_id, 999)
        all_dists = list(self.opp_min_dist.values())
        min_d = min(all_dists)
        max_d = max(all_dists)
        dist_range   = max(1, max_d - min_d)
        norm         = (my_dist - min_d) / dist_range          # 0 = closest, 1 = farthest
        proximity_sc = 1.30 - 0.75 * norm

        # Weakness bonus: weaker opponents are easier to eliminate
        max_str      = max(self.owner_strength.get(o, 0) for o in self.opp_min_dist) or 1
        opp_str      = self.owner_strength.get(opp_id, 0)
        weakness_bonus = max(0.0, 0.30 * (1.0 - opp_str / max_str))

        return max(0.5, proximity_sc + weakness_bonus)

    def is_static(self, planet_id):
        return is_static_planet(self.planet_by_id[planet_id])

    def comet_life(self, planet_id):
        return comet_remaining_life(planet_id, self.comets)

    def source_inventory_left(self, source_id, spent_total):
        return max(0, int(self.planet_by_id[source_id].ships) - spent_total[source_id])

    def plan_shot(self, src_id, target_id, ships):
        ships = int(ships)
        key   = (src_id, target_id, ships)
        if key in self.shot_cache:
            return self.shot_cache[key]
        src    = self.planet_by_id[src_id]
        target = self.planet_by_id[target_id]
        result = aim_with_prediction(
            src, target, ships,
            self.initial_by_id, self.ang_vel, self.comets, self.comet_ids,
        )
        self.shot_cache[key] = result
        return result

    def probe_ship_candidates(self, src_id, target_id, source_cap, hints=()):
        cache = self.probe_candidate_cache
        source_cap       = max(1, int(source_cap))
        normalized_hints = tuple(int(math.ceil(h)) for h in hints if h is not None)
        cache_key = (src_id, target_id, source_cap, normalized_hints)
        if cache_key in cache:
            return cache[cache_key]

        target      = self.planet_by_id[target_id]
        target_ships = max(1, int(math.ceil(target.ships)))

        values = set(range(1, min(6, source_cap) + 1))
        values.update({
            source_cap,
            max(1, source_cap // 2),
            max(1, source_cap // 3),
            min(source_cap, PARTIAL_SOURCE_MIN_SHIPS),
            min(source_cap, target_ships + 1),
            min(source_cap, target_ships + 2),
            min(source_cap, target_ships + 4),
            min(source_cap, target_ships + 8),
        })
        # ── IMPROVED: add SPEED_BRACKET_THRESHOLDS as candidates ─────────────
        for bracket in SPEED_BRACKET_THRESHOLDS:
            if 1 <= bracket <= source_cap:
                values.add(bracket)
                if bracket > 1:
                    values.add(max(1, bracket - 1))

        for hint in normalized_hints:
            base = max(1, min(source_cap, hint))
            for delta in (-3, -2, -1, 0, 1, 2, 3):
                candidate = base + delta
                if 1 <= candidate <= source_cap:
                    values.add(candidate)

        result = sorted(values)
        cache[cache_key] = result
        return result

    def best_probe_aim(
        self, src_id, target_id, source_cap,
        hints=(), min_turn=None, max_turn=None,
        anchor_turn=None, max_anchor_diff=None,
    ):
        cache_key = (
            src_id, target_id, max(1, int(source_cap)),
            tuple(hints), min_turn, max_turn, anchor_turn, max_anchor_diff,
        )
        if cache_key in self.best_probe_cache:
            return self.best_probe_cache[cache_key]

        best = best_key = None
        for ships in self.probe_ship_candidates(src_id, target_id, source_cap, hints=hints):
            aim = self.plan_shot(src_id, target_id, ships)
            if aim is None:
                continue
            angle, turns, dtt, path_target = aim
            if min_turn is not None and turns < min_turn:
                continue
            if max_turn is not None and turns > max_turn:
                continue
            if (
                anchor_turn is not None
                and max_anchor_diff is not None
                and abs(turns - anchor_turn) > max_anchor_diff
            ):
                continue

            key = (abs(turns - anchor_turn), turns, ships) if anchor_turn is not None else (turns, ships)
            if best_key is None or key < best_key:
                best_key = key
                best = (ships, (angle, turns, dtt, path_target))

        self.best_probe_cache[cache_key] = best
        return best

    def reaction_times(self, target_id):
        if target_id in self.reaction_cache:
            return self.reaction_cache[target_id]
        target = self.planet_by_id[target_id]
        my_t = enemy_t = 10 ** 9
        for p in self.my_planets:
            seeded = self.best_probe_aim(p.id, target.id, max(1, int(p.ships)))
            if seeded:
                my_t = min(my_t, seeded[1][1])
        for p in self.enemy_planets:
            seeded = self.best_probe_aim(p.id, target.id, max(1, int(p.ships)))
            if seeded:
                enemy_t = min(enemy_t, seeded[1][1])
        self.reaction_cache[target_id] = (my_t, enemy_t)
        return my_t, enemy_t

    def projected_state(self, target_id, arrival_turn, planned_commitments=None, extra_arrivals=()):
        planned_commitments = planned_commitments or {}
        cutoff = max(1, int(math.ceil(arrival_turn)))
        if not planned_commitments.get(target_id) and not extra_arrivals:
            return state_at_timeline(self.base_timeline[target_id], cutoff)

        arrivals = [
            item for item in self.arrivals_by_planet.get(target_id, []) if item[0] <= cutoff
        ]
        arrivals.extend(item for item in planned_commitments.get(target_id, []) if item[0] <= cutoff)
        arrivals.extend(item for item in extra_arrivals if item[0] <= cutoff)
        target = self.planet_by_id[target_id]
        dyn    = simulate_planet_timeline(target, arrivals, self.player, cutoff)
        return state_at_timeline(dyn, cutoff)

    def projected_timeline(self, target_id, horizon, planned_commitments=None, extra_arrivals=()):
        planned_commitments = planned_commitments or {}
        horizon  = max(1, int(math.ceil(horizon)))
        arrivals = [
            item for item in self.arrivals_by_planet.get(target_id, []) if item[0] <= horizon
        ]
        arrivals.extend(item for item in planned_commitments.get(target_id, []) if item[0] <= horizon)
        arrivals.extend(item for item in extra_arrivals if item[0] <= horizon)
        target = self.planet_by_id[target_id]
        return simulate_planet_timeline(target, arrivals, self.player, horizon)

    def hold_status(self, target_id, planned_commitments=None, horizon=HORIZON):
        planned_commitments = planned_commitments or {}
        tl = (
            self.projected_timeline(target_id, horizon, planned_commitments=planned_commitments)
            if planned_commitments.get(target_id)
            else self.base_timeline[target_id]
        )
        return {
            "keep_needed": tl["keep_needed"],
            "min_owned":   tl["min_owned"],
            "first_enemy": tl["first_enemy"],
            "fall_turn":   tl["fall_turn"],
            "holds_full":  tl["holds_full"],
        }

    def _ownership_search_cap(self, eval_turn):
        productive_cap = self.total_production * max(2, eval_turn + 2)
        return max(32, int(self.total_visible_ships + productive_cap + 32))

    def min_ships_to_own_by(
        self, target_id, eval_turn, attacker_owner,
        arrival_turn=None, planned_commitments=None,
        extra_arrivals=(), upper_bound=None,
    ):
        planned_commitments = planned_commitments or {}
        eval_turn    = max(1, int(math.ceil(eval_turn)))
        arrival_turn = eval_turn if arrival_turn is None else max(1, int(math.ceil(arrival_turn)))
        if arrival_turn > eval_turn:
            return (max(1, int(upper_bound)) + 1) if upper_bound is not None else (self._ownership_search_cap(eval_turn) + 1)

        normalized_extra = tuple(
            (max(1, int(math.ceil(turns))), owner, int(ships))
            for turns, owner, ships in extra_arrivals
            if ships > 0 and max(1, int(math.ceil(turns))) <= eval_turn
        )

        cache_key = None
        if arrival_turn == eval_turn and not planned_commitments.get(target_id) and not normalized_extra:
            cache_key = (target_id, eval_turn, attacker_owner)
            cached    = self.exact_need_cache.get(cache_key)
            if cached is not None:
                return cached

        owner_before, ships_before = self.projected_state(
            target_id, eval_turn, planned_commitments=planned_commitments, extra_arrivals=normalized_extra
        )
        if owner_before == attacker_owner:
            if cache_key:
                self.exact_need_cache[cache_key] = 0
            return 0

        def owns_at(ships):
            ow, _ = self.projected_state(
                target_id, eval_turn,
                planned_commitments=planned_commitments,
                extra_arrivals=normalized_extra + ((arrival_turn, attacker_owner, int(ships)),),
            )
            return ow == attacker_owner

        if upper_bound is not None:
            hi = max(1, int(upper_bound))
            if not owns_at(hi):
                return hi + 1
        else:
            hi = max(1, int(math.ceil(ships_before)) + 1)
            search_cap = self._ownership_search_cap(eval_turn)
            while hi <= search_cap and not owns_at(hi):
                hi *= 2
            if hi > search_cap:
                hi = search_cap
                if not owns_at(hi):
                    return hi + 1

        lo = 1
        while lo < hi:
            mid = (lo + hi) // 2
            if owns_at(mid):
                hi = mid
            else:
                lo = mid + 1

        if cache_key:
            self.exact_need_cache[cache_key] = lo
        return lo

    def min_ships_to_own_at(
        self, target_id, arrival_turn, attacker_owner,
        planned_commitments=None, extra_arrivals=(), upper_bound=None,
    ):
        return self.min_ships_to_own_by(
            target_id, arrival_turn, attacker_owner,
            arrival_turn=arrival_turn,
            planned_commitments=planned_commitments,
            extra_arrivals=extra_arrivals,
            upper_bound=upper_bound,
        )

    def reinforcement_needed_to_hold_until(
        self, planet_id, arrival_turn, hold_until,
        planned_commitments=None, upper_bound=None,
    ):
        planned_commitments = planned_commitments or {}
        target       = self.planet_by_id[planet_id]
        arrival_turn = max(1, int(math.ceil(arrival_turn)))
        hold_until   = max(arrival_turn, int(math.ceil(hold_until)))

        if target.owner != self.player:
            return self.min_ships_to_own_by(
                planet_id, hold_until, self.player,
                arrival_turn=arrival_turn,
                planned_commitments=planned_commitments,
                upper_bound=upper_bound,
            )

        def holds_with_reinforcement(ships):
            timeline = self.projected_timeline(
                planet_id, hold_until,
                planned_commitments=planned_commitments,
                extra_arrivals=((arrival_turn, self.player, int(ships)),),
            )
            return all(
                timeline["owner_at"].get(t) == self.player
                for t in range(arrival_turn, hold_until + 1)
            )

        if upper_bound is not None:
            hi = max(1, int(upper_bound))
            if not holds_with_reinforcement(hi):
                return hi + 1
        else:
            hi = 1
            search_cap = self._ownership_search_cap(hold_until)
            while hi <= search_cap and not holds_with_reinforcement(hi):
                hi *= 2
            if hi > search_cap:
                hi = search_cap
                if not holds_with_reinforcement(hi):
                    return hi + 1

        lo = 1
        while lo < hi:
            mid = (lo + hi) // 2
            if holds_with_reinforcement(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

    def ships_needed_to_capture(self, target_id, arrival_turn, planned_commitments=None, extra_arrivals=()):
        return self.min_ships_to_own_at(
            target_id, arrival_turn, self.player,
            planned_commitments=planned_commitments,
            extra_arrivals=extra_arrivals,
        )


# ============================================================
# Route Optimizer (2-hop routing through allied planets)
# ============================================================

class RouteOptimizer:
    def __init__(self, world):
        self.world = world
        self.cache = {}

    def find_2hop_route(self, src_id, target_id, ships):
        key = (src_id, target_id, ships)
        if key in self.cache:
            return self.cache[key]

        best_route = None
        best_time  = 1e9
        direct     = self.world.plan_shot(src_id, target_id, ships)
        if direct:
            best_time = direct[1]

        for mid in self.world.my_planets:
            if mid.id in (src_id, target_id):
                continue
            leg1 = self.world.plan_shot(src_id, mid.id, ships)
            if not leg1:
                continue
            leg2_seeded = self.world.best_probe_aim(mid.id, target_id, ships)
            if not leg2_seeded:
                continue
            total_time = leg1[1] + leg2_seeded[1][1]
            if total_time < best_time:
                best_time  = total_time
                best_route = {
                    "mid_id":     mid.id,
                    "total":      total_time,
                    "leg1_angle": leg1[0],
                }

        self.cache[key] = best_route
        return best_route


# ============================================================
# Strategy helpers
# ============================================================

def planet_distance(a, b):
    return math.hypot(a.x - b.x, a.y - b.y)


def nearest_sources_to_target(target, sources, top_k):
    if top_k <= 0 or len(sources) <= top_k:
        return sources
    return sorted(sources, key=lambda s: (planet_distance(s, target), -int(s.ships), s.id))[:top_k]


def min_legal_reaction_time(target, sources, world):
    best = 10 ** 9
    for src in sources:
        seeded = world.best_probe_aim(src.id, target.id, max(1, int(src.ships)))
        if seeded:
            best = min(best, seeded[1][1])
    return best


def policy_reaction_times(target_id, policy):
    return policy["reaction_time_map"].get(target_id, (10 ** 9, 10 ** 9))


def candidate_time_valid(target, turns, world, remaining_buffer):
    if turns > world.remaining_steps - remaining_buffer:
        return False
    if target.id in world.comet_ids:
        life = world.comet_life(target.id)
        if turns >= life or turns > COMET_MAX_CHASE_TURNS:
            return False
    return True


def stacked_enemy_proactive_keep(planet, world):
    threats = []
    for enemy in world.enemy_planets:
        seeded = world.best_probe_aim(enemy.id, planet.id, max(1, int(enemy.ships)))
        if seeded is None:
            continue
        _, aim = seeded
        eta = aim[1]
        if eta > MULTI_ENEMY_PROACTIVE_HORIZON:
            continue
        threats.append((eta, int(enemy.ships)))
    if not threats:
        return 0
    threats.sort()
    best_stacked = left = running = 0
    for right in range(len(threats)):
        running += threats[right][1]
        while threats[right][0] - threats[left][0] > MULTI_ENEMY_STACK_WINDOW:
            running -= threats[left][1]
            left += 1
        best_stacked = max(best_stacked, running)
    return int(best_stacked * MULTI_ENEMY_PROACTIVE_RATIO)


def swarm_eta_tolerance(options, target, world):
    if len(options) >= 3:
        return THREE_SOURCE_ETA_TOLERANCE
    if target.owner not in (-1, world.player):
        return HOSTILE_SWARM_ETA_TOLERANCE
    return MULTI_SOURCE_ETA_TOLERANCE


def detect_enemy_crashes(world):
    crashes = []
    for target_id, arrivals in world.arrivals_by_planet.items():
        enemy_events = sorted(
            [
                (int(math.ceil(eta)), owner, int(ships))
                for eta, owner, ships in arrivals
                if owner not in (-1, world.player) and ships > 0
            ]
        )
        for i in range(len(enemy_events)):
            eta_a, owner_a, ships_a = enemy_events[i]
            for j in range(i + 1, len(enemy_events)):
                eta_b, owner_b, ships_b = enemy_events[j]
                if owner_a == owner_b:
                    continue
                if abs(eta_a - eta_b) > CRASH_EXPLOIT_ETA_WINDOW:
                    break
                if ships_a + ships_b < CRASH_EXPLOIT_MIN_TOTAL_SHIPS:
                    continue
                crashes.append({
                    "target_id":  target_id,
                    "crash_turn": max(eta_a, eta_b),
                    "owners":     (owner_a, owner_b),
                    "ships":      (ships_a, ships_b),
                })
    return crashes


# ── Policy state ─────────────────────────────────────────────────────────────

def build_policy_state(world, deadline=None):
    def expired():
        return deadline is not None and time.perf_counter() > deadline

    indirect_wealth_map = {}
    for target_id, (friendly, neutral, enemy) in world.indirect_feature_map.items():
        indirect_wealth_map[target_id] = (
            friendly * INDIRECT_FRIENDLY_WEIGHT
            + neutral * INDIRECT_NEUTRAL_WEIGHT
            + enemy   * INDIRECT_ENEMY_WEIGHT
        )

    reserve       = {}
    attack_budget = {}
    reaction_time_map = {}

    for target in world.planets:
        if expired():
            break
        if target.owner == world.player:
            continue
        my_sources    = nearest_sources_to_target(target, world.my_planets,     REACTION_SOURCE_TOP_K_MY)
        enemy_sources = nearest_sources_to_target(target, world.enemy_planets,  REACTION_SOURCE_TOP_K_ENEMY)
        my_t          = min_legal_reaction_time(target, my_sources,    world)
        enemy_t       = min_legal_reaction_time(target, enemy_sources, world)
        reaction_time_map[target.id] = (my_t, enemy_t)

    # ── IMPROVED: threat-aware reserve computation ───────────────────────────
    for planet in world.my_planets:
        if expired():
            break
        exact_keep = world.keep_needed_map.get(planet.id, 0)

        # Proactive keep from nearest enemies
        proactive_keep = 0
        for enemy in nearest_sources_to_target(planet, world.enemy_planets, PROACTIVE_ENEMY_TOP_K):
            enemy_aim = world.plan_shot(enemy.id, planet.id, max(1, int(enemy.ships)))
            if enemy_aim is None:
                continue
            if enemy_aim[1] <= PROACTIVE_DEFENSE_HORIZON:
                proactive_keep = max(proactive_keep, int(enemy.ships * PROACTIVE_DEFENSE_RATIO))
        proactive_keep = max(proactive_keep, stacked_enemy_proactive_keep(planet, world))

        # ── NEW: threat-map reserve ──────────────────────────────────────────
        threat_based_keep = int(world.threat_map.get(planet.id, 0) * 1.2)

        # Frontier specialisation
        dist_to_enemy = nearest_distance_to_set(planet.x, planet.y, world.enemy_planets)
        base_reserve  = max(exact_keep, proactive_keep, threat_based_keep)

        if dist_to_enemy < 22:
            # Front-line fortress: keep more
            specialized_reserve = max(base_reserve, int(planet.ships * 0.50))
        elif dist_to_enemy > 55:
            # Safe rear: can be milked aggressively
            specialized_reserve = int(base_reserve * 0.55)
        else:
            specialized_reserve = base_reserve

        # ── NEW: avalanche mode — drain rear planets ─────────────────────────
        is_avalanche = (
            world.enemy_prod > 0
            and world.my_prod / world.enemy_prod > AVALANCHE_PROD_RATIO
            and dist_to_enemy > 40
        )
        if is_avalanche:
            specialized_reserve = int(base_reserve * 0.40)

        reserve[planet.id]       = min(int(planet.ships), specialized_reserve)
        attack_budget[planet.id] = max(0, int(planet.ships) - reserve[planet.id])

    return {
        "indirect_wealth_map": indirect_wealth_map,
        "reserve":             reserve,
        "attack_budget":       attack_budget,
        "reaction_time_map":   reaction_time_map,
    }


def build_modes(world):
    domination  = (world.my_total - world.enemy_total) / max(1, world.my_total + world.enemy_total)
    is_behind   = domination < BEHIND_DOMINATION
    is_ahead    = domination > AHEAD_DOMINATION
    is_dominating = is_ahead or (
        world.max_enemy_strength > 0 and world.my_total > world.max_enemy_strength * 1.25
    )
    is_finishing = (
        domination > FINISHING_DOMINATION
        and world.my_prod > world.enemy_prod * FINISHING_PROD_RATIO
        and world.step > 100
    )

    # ── NEW: avalanche and elimination ───────────────────────────────────────
    is_avalanche = (
        world.enemy_prod > 0
        and world.my_prod / world.enemy_prod > AVALANCHE_PROD_RATIO
    )
    # Elimination surge: any enemy whose total strength is tiny vs ours
    is_elimination_surge = any(
        world.owner_strength.get(opp_id, 0) < world.my_total * ELIMINATION_STRENGTH_RATIO
        for opp_id in world.opp_planets
    )

    attack_margin_mult = 1.0
    if is_ahead:
        attack_margin_mult += AHEAD_ATTACK_MARGIN_BONUS
    if is_behind:
        attack_margin_mult -= BEHIND_ATTACK_MARGIN_PENALTY
    if is_finishing:
        attack_margin_mult += FINISHING_ATTACK_MARGIN_BONUS

    return {
        "domination":           domination,
        "is_behind":            is_behind,
        "is_ahead":             is_ahead,
        "is_dominating":        is_dominating,
        "is_finishing":         is_finishing,
        "is_avalanche":         is_avalanche,
        "is_elimination_surge": is_elimination_surge,
        "attack_margin_mult":   attack_margin_mult,
    }


def is_safe_neutral(target, policy):
    if target.owner != -1:
        return False
    my_t, enemy_t = policy_reaction_times(target.id, policy)
    return my_t <= enemy_t - SAFE_NEUTRAL_MARGIN


def is_contested_neutral(target, policy):
    if target.owner != -1:
        return False
    my_t, enemy_t = policy_reaction_times(target.id, policy)
    return abs(my_t - enemy_t) <= CONTESTED_NEUTRAL_MARGIN


def opening_filter(target, arrival_turns, needed, src_available, world, policy):
    if not world.is_opening or target.owner != -1:
        return False
    if target.id in world.comet_ids:
        return False
    if world.is_static(target.id):
        return False
    my_t, enemy_t = policy_reaction_times(target.id, policy)
    reaction_gap   = enemy_t - my_t
    if (
        target.production >= SAFE_OPENING_PROD_THRESHOLD
        and arrival_turns <= SAFE_OPENING_TURN_LIMIT
        and reaction_gap >= SAFE_NEUTRAL_MARGIN
    ):
        return False
    if world.is_four_player:
        affordable = needed <= max(PARTIAL_SOURCE_MIN_SHIPS, int(src_available * FOUR_PLAYER_ROTATING_SEND_RATIO))
        if affordable and arrival_turns <= FOUR_PLAYER_ROTATING_TURN_LIMIT and reaction_gap >= FOUR_PLAYER_ROTATING_REACTION_GAP:
            return False
        return True
    return arrival_turns > ROTATING_OPENING_MAX_TURNS or target.production <= ROTATING_OPENING_LOW_PROD


# ── IMPROVED: NPV-based target value ─────────────────────────────────────────

def target_value(target, arrival_turns, mission, world, modes, policy):
    remaining = max(1, world.remaining_steps - arrival_turns)

    if target.id in world.comet_ids:
        life = world.comet_life(target.id)
        remaining = max(0, min(remaining, life - arrival_turns))
        if remaining <= 0:
            return -1.0

    # ── Base value: production × remaining turns ──────────────────────────
    value = target.production * remaining

    # ── IMPROVED: NPV compounding bonus ──────────────────────────────────
    # A planet captured now vs N turns later generates N*production extra ships.
    # Those ships compound into future attacks. We model this as a bonus that
    # decays for planets that are already captured quickly.
    compounding_turns = max(0, NPV_COMPOUNDING_CAP_TURN - arrival_turns)
    npv_bonus = target.production * compounding_turns * NPV_COMPOUNDING_RATE
    value += npv_bonus

    # ── IMPROVED: early rush bonus (extra multiplier for very fast captures) ─
    if arrival_turns < EARLY_RUSH_TURNS:
        rush_bonus = target.production * (EARLY_RUSH_TURNS - arrival_turns) * EARLY_RUSH_MULT
        value += rush_bonus

    # Neighbourhood value
    value += policy["indirect_wealth_map"][target.id] * remaining * INDIRECT_VALUE_SCALE

    # Static / rotating multiplier
    if world.is_static(target.id):
        value *= STATIC_NEUTRAL_VALUE_MULT if target.owner == -1 else STATIC_HOSTILE_VALUE_MULT
    else:
        value *= ROTATING_OPENING_VALUE_MULT if world.is_opening else 1.0

    # Enemy planet premium
    if target.owner not in (-1, world.player):
        base_mult   = OPENING_HOSTILE_TARGET_VALUE_MULT if world.is_opening else HOSTILE_TARGET_VALUE_MULT
        opp_priority = world.opponent_priority(target.owner)
        value *= base_mult * opp_priority

        # ── IMPROVED: elimination surge — massive bonus when enemy is weak ──
        if modes["is_elimination_surge"]:
            opp_str = world.owner_strength.get(target.owner, 0)
            if opp_str < world.my_total * ELIMINATION_STRENGTH_RATIO:
                value *= ELIMINATION_VALUE_MULT

    # Neutral planet categorisation
    if target.owner == -1:
        if is_safe_neutral(target, policy):
            value *= SAFE_NEUTRAL_VALUE_MULT
        elif is_contested_neutral(target, policy):
            value *= CONTESTED_NEUTRAL_VALUE_MULT

            # ── IMPROVED: denial bonus for contested neutrals ──────────────
            # If the enemy is racing us here, denying them is worth extra.
            _, enemy_t = policy_reaction_times(target.id, policy)
            if enemy_t < 10 ** 8:
                denial_val = target.production * min(remaining, world.remaining_steps) * DENIAL_BONUS_MULT
                value += denial_val

        if world.is_early:
            value *= EARLY_NEUTRAL_VALUE_MULT

    if target.id in world.comet_ids:
        value *= COMET_VALUE_MULT

    # Mission type multipliers
    if mission == "snipe":
        value *= SNIPE_VALUE_MULT
    elif mission == "swarm":
        value *= SWARM_VALUE_MULT
    elif mission == "reinforce":
        value *= REINFORCE_VALUE_MULT
    elif mission == "crash_exploit":
        value *= CRASH_EXPLOIT_VALUE_MULT
    elif mission == "interdiction":
        value *= INTERDICTION_VALUE_MULT

    # Late-game ship immediacy
    if world.is_late:
        value += max(0, target.ships) * LATE_IMMEDIATE_SHIP_VALUE
        if target.owner not in (-1, world.player):
            enemy_strength = world.owner_strength.get(target.owner, 0)
            if enemy_strength <= WEAK_ENEMY_THRESHOLD:
                value += ELIMINATION_BONUS

    if modes["is_finishing"] and target.owner not in (-1, world.player):
        value *= FINISHING_HOSTILE_VALUE_MULT
    if modes["is_behind"] and target.owner == -1 and not world.is_static(target.id):
        value *= BEHIND_ROTATING_NEUTRAL_VALUE_MULT
    if modes["is_behind"] and target.owner == -1 and is_safe_neutral(target, policy):
        value *= 1.08
    if modes["is_dominating"] and target.owner == -1 and is_contested_neutral(target, policy):
        value *= 0.92

    return value


def reinforce_value(target, hold_until, world, policy):
    saved_turns = max(1, world.remaining_steps - hold_until)
    value = target.production * saved_turns + max(0, target.ships) * DEFENSE_SHIP_VALUE
    if world.enemy_planets and nearest_distance_to_set(target.x, target.y, world.enemy_planets) < 22:
        value *= DEFENSE_FRONTIER_SCORE_MULT
    value += policy["indirect_wealth_map"][target.id] * saved_turns * INDIRECT_VALUE_SCALE * 0.35
    return value * REINFORCE_VALUE_MULT


# ── IMPROVED: preferred_send with speed-bracket exploitation ─────────────────

def preferred_send(target, base_needed, arrival_turns, src_available, world, modes, policy, src=None):
    send   = max(base_needed, int(math.ceil(base_needed * modes["attack_margin_mult"])))
    margin = 0

    if target.owner == -1:
        margin += min(NEUTRAL_MARGIN_CAP, NEUTRAL_MARGIN_BASE + target.production * NEUTRAL_MARGIN_PROD_WEIGHT)
    else:
        margin += min(HOSTILE_MARGIN_CAP, HOSTILE_MARGIN_BASE + target.production * HOSTILE_MARGIN_PROD_WEIGHT)
        reinforce_est = 0
        for ep in world.opp_planets.get(target.owner, []):
            if ep.id == target.id:
                continue
            ep_aim = world.plan_shot(ep.id, target.id, max(1, int(ep.ships)))
            if ep_aim and ep_aim[1] <= arrival_turns + HOSTILE_REINFORCE_HORIZON:
                reinforce_est += max(0, int(ep.ships) - 3)
        margin += min(HOSTILE_REINFORCE_CAP, int(reinforce_est * HOSTILE_REINFORCE_RATIO))

    if world.is_static(target.id):
        margin += STATIC_TARGET_MARGIN
    if is_contested_neutral(target, policy):
        margin += CONTESTED_TARGET_MARGIN
    if world.is_four_player:
        margin += FOUR_PLAYER_TARGET_MARGIN
    if arrival_turns > LONG_TRAVEL_MARGIN_START:
        margin += min(LONG_TRAVEL_MARGIN_CAP, arrival_turns // LONG_TRAVEL_MARGIN_DIVISOR)
    if target.id in world.comet_ids:
        margin = max(0, margin - COMET_MARGIN_RELIEF)
    if modes["is_finishing"] and target.owner not in (-1, world.player):
        margin += FINISHING_HOSTILE_SEND_BONUS

    final_send    = min(src_available, send + margin)
    current_turns = arrival_turns

    # ── IMPROVED: speed-bracket exploitation ─────────────────────────────────
    # For each bracket threshold above our current send count, check if jumping
    # to that bracket reduces ETA and whether the saved turns justify the extra ships.
    if src is not None and final_send < src_available:
        for bracket in SPEED_BRACKET_THRESHOLDS:
            if bracket <= final_send:
                continue
            if bracket > src_available:
                break
            est = estimate_arrival(
                src.x, src.y, src.radius,
                target.x, target.y, target.radius,
                bracket
            )
            if est is None:
                continue
            saved_turns = current_turns - est[1]
            if saved_turns <= 0:
                continue
            # Worth it if saved production > extra ships spent
            extra_ships    = bracket - final_send
            gained_prod    = target.production * saved_turns
            # Also worth it if it lets us arrive before a known enemy transit
            transit_bonus = 0
            for eta, owner, ships in world.enemy_transits.get(target.id, []):
                if current_turns >= eta and est[1] < eta:
                    transit_bonus = ships + target.production * 5  # huge bonus
            if gained_prod + transit_bonus >= extra_ships * 0.8:
                final_send    = bracket
                current_turns = est[1]
                if saved_turns <= 1:
                    break  # one-turn improvement is fine, don't over-commit

    return min(src_available, final_send)


def apply_score_modifiers(base_score, target, mission, world):
    score = base_score
    if world.is_static(target.id):
        score *= STATIC_TARGET_SCORE_MULT
    if world.is_early and target.owner == -1 and world.is_static(target.id):
        score *= EARLY_STATIC_NEUTRAL_SCORE_MULT
    if world.is_four_player and target.owner == -1 and not world.is_static(target.id):
        score *= FOUR_PLAYER_ROTATING_NEUTRAL_SCORE_MULT
    if (
        len(world.static_neutral_planets) >= DENSE_STATIC_NEUTRAL_COUNT
        and target.owner == -1 and not world.is_static(target.id)
    ):
        score *= DENSE_ROTATING_NEUTRAL_SCORE_MULT
    if mission == "snipe":
        score *= SNIPE_SCORE_MULT
    elif mission == "swarm":
        score *= SWARM_SCORE_MULT
    elif mission == "crash_exploit":
        score *= CRASH_EXPLOIT_SCORE_MULT
    elif mission == "interdiction":
        score *= 1.35   # NEW — interdiction has a score premium
    return score


# ── settle_plan and settle_reinforce_plan (unchanged logic, same API) ─────────

def settle_plan(
    src, target, src_cap, send_guess, world,
    planned_commitments, modes, policy,
    mission="capture", eval_turn_fn=None,
    anchor_turn=None, anchor_tolerance=None,
    max_iter=4, router=None,
):
    if src_cap < 1:
        return None
    seed_hint         = max(1, min(src_cap, int(send_guess)))
    eval_turn_fn      = eval_turn_fn or (lambda turns: turns)
    anchor_tolerance  = (
        anchor_tolerance if anchor_tolerance is not None
        else (1 if mission == "snipe" else None)
    )
    tested      = {}
    tested_order = []

    def evaluate(send):
        send   = max(1, min(src_cap, int(send)))
        cached = tested.get(send)
        if cached is not None or send in tested:
            return cached

        aim = world.plan_shot(src.id, target.id, send)
        if aim is None and router is not None:
            route = router.find_2hop_route(src.id, target.id, send)
            if route:
                aim = (route["leg1_angle"], route["total"], 0, None)
        if aim is None:
            tested[send] = None
            return None

        angle, turns, _, _ = aim
        if mission == "crash_exploit" and anchor_turn is not None and turns < anchor_turn:
            tested[send] = None
            return None
        raw_eval = int(math.ceil(eval_turn_fn(turns)))
        if raw_eval < turns:
            tested[send] = None
            return None
        eval_turn = raw_eval
        need = world.min_ships_to_own_by(
            target.id, eval_turn, world.player,
            arrival_turn=turns,
            planned_commitments=planned_commitments,
            upper_bound=src_cap,
        )
        if need <= 0 or need > src_cap:
            tested[send] = None
            return None

        if mission in ("snipe", "crash_exploit"):
            desired = need
        elif mission == "rescue":
            desired = min(
                src_cap,
                max(need, need + DEFENSE_SEND_MARGIN_BASE + target.production * DEFENSE_SEND_MARGIN_PROD_WEIGHT),
            )
        else:
            desired = min(
                src_cap,
                max(need, preferred_send(target, need, turns, src_cap, world, modes, policy, src=src)),
            )

        result = (angle, turns, eval_turn, need, send, desired)
        tested[send] = result
        tested_order.append(send)
        return result

    initial_candidates = sorted(
        world.probe_ship_candidates(src.id, target.id, src_cap, hints=(seed_hint,)),
        key=lambda s: (abs(s - seed_hint), s),
    )
    current_send = None
    for seed in initial_candidates:
        result = evaluate(seed)
        if result is None:
            continue
        if (
            anchor_turn is not None and anchor_tolerance is not None
            and abs(result[1] - anchor_turn) > anchor_tolerance
        ):
            continue
        current_send = seed
        break

    if current_send is None:
        return None

    for _ in range(max_iter):
        result = evaluate(current_send)
        if result is None:
            break
        angle, turns, eval_turn, need, actual_send, desired = result
        if desired == actual_send:
            if anchor_turn is not None and anchor_tolerance is not None and abs(turns - anchor_turn) > anchor_tolerance:
                return None
            if mission == "rescue" and turns > eval_turn:
                return None
            return angle, turns, eval_turn, need, actual_send
        next_send = max(1, min(src_cap, int(desired)))
        if next_send in tested:
            current_send = next_send
            break
        current_send = next_send

    candidate_sends = sorted(
        [s for s in tested_order if tested.get(s) is not None],
        key=lambda s: (
            0 if mission != "snipe" or anchor_turn is None else abs(tested[s][1] - anchor_turn),
            abs(s - seed_hint),
            tested[s][1],
            s,
        ),
    )
    seen = set()
    for send in candidate_sends:
        if send in seen:
            continue
        seen.add(send)
        result = tested.get(send)
        if result is None:
            continue
        angle, turns, eval_turn, need, actual_send, _ = result
        if actual_send < need:
            continue
        if anchor_turn is not None and anchor_tolerance is not None and abs(turns - anchor_turn) > anchor_tolerance:
            continue
        if mission == "rescue" and turns > eval_turn:
            continue
        return angle, turns, eval_turn, need, actual_send
    return None


def settle_reinforce_plan(
    src, target, src_cap, send_guess, world,
    planned_commitments, hold_until, max_arrival_turn, max_iter=4,
):
    if src_cap < 1:
        return None
    seed_hint    = max(1, min(src_cap, int(send_guess)))
    tested       = {}
    tested_order = []

    def evaluate(send):
        send   = max(1, min(src_cap, int(send)))
        cached = tested.get(send)
        if cached is not None or send in tested:
            return cached
        aim = world.plan_shot(src.id, target.id, send)
        if aim is None:
            tested[send] = None
            return None
        angle, turns, _, _ = aim
        if turns > max_arrival_turn:
            tested[send] = None
            return None
        need = world.reinforcement_needed_to_hold_until(
            target.id, turns, hold_until,
            planned_commitments=planned_commitments,
            upper_bound=src_cap,
        )
        if need <= 0 or need > src_cap:
            tested[send] = None
            return None
        desired = min(src_cap, need + REINFORCE_SAFETY_MARGIN)
        result  = (angle, turns, hold_until, need, send, desired)
        tested[send] = result
        tested_order.append(send)
        return result

    initial_candidates = sorted(
        world.probe_ship_candidates(src.id, target.id, src_cap, hints=(seed_hint,)),
        key=lambda s: (abs(s - seed_hint), s),
    )
    current_send = None
    for seed in initial_candidates:
        result = evaluate(seed)
        if result:
            current_send = seed
            break
    if current_send is None:
        return None

    for _ in range(max_iter):
        result = evaluate(current_send)
        if result is None:
            break
        angle, turns, eval_turn, need, actual_send, desired = result
        if desired == actual_send:
            return angle, turns, eval_turn, need, actual_send
        next_send = max(1, min(src_cap, int(desired)))
        if next_send in tested:
            current_send = next_send
            break
        current_send = next_send

    for send in sorted([s for s in tested_order if tested.get(s)], key=lambda s: (abs(s - seed_hint), tested[s][1], s)):
        result = tested.get(send)
        if result and result[4] >= result[3] and result[1] <= max_arrival_turn:
            return result[0], result[1], result[2], result[3], result[4]
    return None


# ── Mission builders ──────────────────────────────────────────────────────────

def build_snipe_mission(src, target, src_available, world, planned_commitments, modes, policy, router=None):
    if target.owner != -1:
        return None
    enemy_etas = sorted({
        int(math.ceil(eta))
        for eta, owner, ships in world.arrivals_by_planet.get(target.id, [])
        if owner not in (-1, world.player) and ships > 0
    })
    if not enemy_etas:
        return None

    best = None
    for enemy_eta in enemy_etas[:3]:
        seeded = world.best_probe_aim(
            src.id, target.id, src_available,
            hints=(int(target.ships) + 1, int(target.ships) + 8),
            anchor_turn=enemy_eta, max_anchor_diff=1,
        )
        if seeded is None:
            continue
        probe, rough = seeded
        sync_turn = max(rough[1], enemy_eta)
        if target.id in world.comet_ids:
            life = world.comet_life(target.id)
            if sync_turn >= life or sync_turn > COMET_MAX_CHASE_TURNS:
                continue
        plan = settle_plan(
            src, target, src_available, probe, world, planned_commitments,
            modes, policy, mission="snipe",
            eval_turn_fn=lambda t, e=enemy_eta: max(t, e),
            anchor_turn=enemy_eta, router=router,
        )
        if plan is None:
            continue
        angle, turns, sync_turn2, need, send_pref = plan
        if target.id in world.comet_ids:
            life = world.comet_life(target.id)
            if sync_turn2 >= life or sync_turn2 > COMET_MAX_CHASE_TURNS:
                continue
        value = target_value(target, sync_turn2, "snipe", world, modes, policy)
        if value <= 0:
            continue
        score = apply_score_modifiers(
            value / (send_pref + sync_turn2 * SNIPE_COST_TURN_WEIGHT + 1.0),
            target, "snipe", world,
        )
        option = ShotOption(
            score=score, src_id=src.id, target_id=target.id,
            angle=angle, turns=turns, needed=need, send_cap=send_pref,
            mission="snipe", anchor_turn=enemy_eta,
        )
        m = Mission(kind="snipe", score=score, target_id=target.id, turns=sync_turn2, options=[option])
        if best is None or m.score > best.score:
            best = m
    return best


def build_shadow_snipe_missions(src, target, src_available, world, planned_commitments, modes, policy, router=None):
    if target.owner != -1:
        return None
    best_opp = None
    max_intent = 0
    for opp_id in world.opp_planets.keys():
        intent = world.predict_enemy_intent(opp_id).get(target.id, 0)
        if intent > max_intent:
            max_intent = intent
            best_opp   = opp_id
    if best_opp is None or max_intent < SHADOW_SNIPE_INTENT_THRESHOLD:
        return None
    opp_srcs    = world.opp_planets[best_opp]
    best_opp_src = min(opp_srcs, key=lambda s: dist(s.x, s.y, target.x, target.y))
    opp_d       = dist(best_opp_src.x, best_opp_src.y, target.x, target.y)
    target_eta  = int(math.ceil(opp_d / 2.0)) + 1
    plan = settle_plan(
        src, target, src_available, int(target.ships) + 2,
        world, planned_commitments, modes, policy, mission="snipe",
        anchor_turn=target_eta, anchor_tolerance=1, router=router,
    )
    if plan is None:
        return None
    angle, turns, sync_turn, need, send_pref = plan
    score = target_value(target, sync_turn, "snipe", world, modes, policy) * 0.8
    score /= (send_pref + sync_turn * SNIPE_COST_TURN_WEIGHT + 1.0)
    option = ShotOption(
        score=score, src_id=src.id, target_id=target.id,
        angle=angle, turns=turns, needed=need, send_cap=send_pref,
        mission="shadow_snipe", anchor_turn=target_eta,
    )
    return Mission(kind="shadow_snipe", score=score, target_id=target.id, turns=sync_turn, options=[option])


def build_rescue_missions(world, policy, planned_commitments, modes, router=None):
    missions = []
    for target in world.my_planets:
        fall_turn = world.fall_turn_map.get(target.id)
        if fall_turn is None or fall_turn > DEFENSE_LOOKAHEAD_TURNS:
            continue
        for src in world.my_planets:
            if src.id == target.id:
                continue
            src_available = policy["attack_budget"].get(src.id, 0)
            if src_available < PARTIAL_SOURCE_MIN_SHIPS:
                continue
            seeded = world.best_probe_aim(
                src.id, target.id, src_available,
                hints=(target.production + DEFENSE_SEND_MARGIN_BASE + 2,),
                max_turn=fall_turn,
            )
            if seeded is None:
                continue
            probe, _ = seeded
            plan = settle_plan(
                src, target, src_available, probe, world, planned_commitments,
                modes, policy, mission="rescue",
                eval_turn_fn=lambda _t, ft=fall_turn: ft,
                anchor_turn=fall_turn, router=router,
            )
            if plan is None:
                continue
            angle, turns, _, need, send_pref = plan
            saved = max(1, world.remaining_steps - fall_turn)
            value = target.production * saved + max(0, target.ships) * DEFENSE_SHIP_VALUE
            if world.enemy_planets and nearest_distance_to_set(target.x, target.y, world.enemy_planets) < 22:
                value *= DEFENSE_FRONTIER_SCORE_MULT
            score = value / (send_pref + turns * DEFENSE_COST_TURN_WEIGHT + 1.0)
            option = ShotOption(
                score=score, src_id=src.id, target_id=target.id,
                angle=angle, turns=turns, needed=need, send_cap=send_pref,
                mission="rescue", anchor_turn=fall_turn,
            )
            missions.append(Mission(kind="rescue", score=score, target_id=target.id, turns=fall_turn, options=[option]))
    return missions


def build_recapture_missions(world, policy, planned_commitments, modes, router=None):
    missions = []
    for target in world.my_planets:
        fall_turn = world.fall_turn_map.get(target.id)
        if fall_turn is None or fall_turn > DEFENSE_LOOKAHEAD_TURNS:
            continue
        for src in world.my_planets:
            if src.id == target.id:
                continue
            src_available = policy["attack_budget"].get(src.id, 0)
            if src_available < PARTIAL_SOURCE_MIN_SHIPS:
                continue
            seeded = world.best_probe_aim(
                src.id, target.id, src_available,
                hints=(target.production + DEFENSE_SEND_MARGIN_BASE + 2,),
                min_turn=fall_turn + 1, max_turn=fall_turn + RECAPTURE_LOOKAHEAD_TURNS,
            )
            if seeded is None:
                continue
            probe, _ = seeded
            plan = settle_plan(
                src, target, src_available, probe, world, planned_commitments,
                modes, policy, mission="capture", router=router,
            )
            if plan is None:
                continue
            angle, turns, _, need, send_pref = plan
            if turns <= fall_turn or turns - fall_turn > RECAPTURE_LOOKAHEAD_TURNS:
                continue
            saved = max(1, world.remaining_steps - turns)
            value = (
                RECAPTURE_PRODUCTION_WEIGHT * target.production * saved
                + RECAPTURE_IMMEDIATE_WEIGHT * max(0, target.ships)
            )
            if world.enemy_planets and nearest_distance_to_set(target.x, target.y, world.enemy_planets) < 22:
                value *= RECAPTURE_FRONTIER_MULT
            value *= RECAPTURE_VALUE_MULT
            score = value / (send_pref + turns * RECAPTURE_COST_TURN_WEIGHT + 1.0)
            option = ShotOption(
                score=score, src_id=src.id, target_id=target.id,
                angle=angle, turns=turns, needed=need, send_cap=send_pref,
                mission="recapture", anchor_turn=fall_turn,
            )
            missions.append(Mission(kind="recapture", score=score, target_id=target.id, turns=turns, options=[option]))
    return missions


def build_reinforce_missions(world, policy, planned_commitments, modes, inventory_left_fn):
    if not REINFORCE_ENABLED:
        return []
    missions = []
    if world.remaining_steps < REINFORCE_MIN_FUTURE_TURNS:
        return missions
    for target in world.my_planets:
        fall_turn = world.fall_turn_map.get(target.id)
        if fall_turn is None:
            continue
        if target.production < REINFORCE_MIN_PRODUCTION:
            continue
        hold_until      = min(HORIZON, fall_turn + REINFORCE_HOLD_LOOKAHEAD)
        max_arrival_turn = min(fall_turn, REINFORCE_MAX_TRAVEL_TURNS)
        for src in world.my_planets:
            if src.id == target.id:
                continue
            budget     = inventory_left_fn(src.id)
            source_cap = min(budget, int(src.ships * REINFORCE_MAX_SOURCE_FRACTION))
            if source_cap < PARTIAL_SOURCE_MIN_SHIPS:
                continue
            seeded = world.best_probe_aim(
                src.id, target.id, source_cap,
                hints=(target.production + REINFORCE_SAFETY_MARGIN + 2,),
                max_turn=max_arrival_turn,
            )
            if seeded is None:
                continue
            probe, _ = seeded
            plan = settle_reinforce_plan(
                src, target, source_cap, probe, world, planned_commitments,
                hold_until, max_arrival_turn,
            )
            if plan is None:
                continue
            angle, turns, _, need, send_pref = plan
            value = reinforce_value(target, hold_until, world, policy)
            score = value / (send_pref + turns * REINFORCE_COST_TURN_WEIGHT + 1.0)
            option = ShotOption(
                score=score, src_id=src.id, target_id=target.id,
                angle=angle, turns=turns, needed=need, send_cap=send_pref,
                mission="reinforce", anchor_turn=hold_until,
            )
            missions.append(Mission(kind="reinforce", score=score, target_id=target.id, turns=fall_turn, options=[option]))
    return missions


def build_crash_exploit_missions(world, policy, planned_commitments, modes, router=None):
    # Extended to 2-player as well (was 4-player only)
    if not CRASH_EXPLOIT_ENABLED:
        return []
    missions = []
    for crash in detect_enemy_crashes(world):
        target = world.planet_by_id[crash["target_id"]]
        if target.owner == world.player:
            continue
        desired_arrival = crash["crash_turn"] + CRASH_EXPLOIT_POST_CRASH_DELAY
        for src in world.my_planets:
            src_available = policy["attack_budget"].get(src.id, 0)
            if src_available < PARTIAL_SOURCE_MIN_SHIPS:
                continue
            seeded = world.best_probe_aim(
                src.id, target.id, src_available,
                hints=(12, int(target.ships) + 1),
                anchor_turn=desired_arrival, max_anchor_diff=CRASH_EXPLOIT_ETA_WINDOW,
            )
            if seeded is None:
                continue
            probe, _ = seeded
            plan = settle_plan(
                src, target, src_available, probe, world, planned_commitments,
                modes, policy, mission="crash_exploit",
                eval_turn_fn=lambda t, d=desired_arrival: max(t, d),
                anchor_turn=desired_arrival, anchor_tolerance=CRASH_EXPLOIT_ETA_WINDOW,
                router=router,
            )
            if plan is None:
                continue
            angle, turns, _, need, send_pref = plan
            if not candidate_time_valid(target, turns, world, LATE_CAPTURE_BUFFER):
                continue
            value = target_value(target, turns, "crash_exploit", world, modes, policy)
            if value <= 0:
                continue
            score = apply_score_modifiers(
                value / (send_pref + turns * SNIPE_COST_TURN_WEIGHT + 1.0),
                target, "crash_exploit", world,
            )
            option = ShotOption(
                score=score, src_id=src.id, target_id=target.id,
                angle=angle, turns=turns, needed=need, send_cap=send_pref,
                mission="crash_exploit", anchor_turn=desired_arrival,
            )
            missions.append(Mission(kind="crash_exploit", score=score, target_id=target.id, turns=turns, options=[option]))
    return missions


# ── NEW: Interdiction missions ────────────────────────────────────────────────
# Race enemy in-transit fleets to neutral planets and arrive first.
# This denies the enemy resources AND grants us a contested neutral for free.

def build_interdiction_missions(world, policy, planned_commitments, modes, router=None):
    """
    Detect enemy fleets heading for neutral planets.  For each such case, try
    to dispatch one of our planets to arrive INTERDICTION_LEAD_TURNS before the
    enemy.  A successful interdiction gives us the planet before the enemy lands
    and we can pile ships on — the enemy fleet wasted its travel time.
    """
    missions = []
    if world.is_very_late:
        return missions

    # Gather all enemy transits headed to neutral planets
    for target_id, transits in world.enemy_transits.items():
        target = world.planet_by_id[target_id]
        if target.owner != -1:          # only neutral interceptions
            continue
        if target.production < INTERDICTION_MIN_PROD:
            continue
        if target.id in world.comet_ids:
            continue

        # Take the earliest (most urgent) enemy transit
        valid = [(eta, owner, ships) for eta, owner, ships in transits if ships >= INTERDICTION_MIN_FLEET_SHIPS]
        if not valid:
            continue
        valid.sort()
        earliest_eta, _, enemy_ships = valid[0]

        # We want to arrive at most INTERDICTION_LEAD_TURNS before enemy
        must_arrive_by = max(1, earliest_eta - INTERDICTION_LEAD_TURNS)
        if must_arrive_by > world.remaining_steps - LATE_CAPTURE_BUFFER:
            continue

        for src in world.my_planets:
            src_available = policy["attack_budget"].get(src.id, 0)
            if src_available < PARTIAL_SOURCE_MIN_SHIPS:
                continue

            seeded = world.best_probe_aim(
                src.id, target.id, src_available,
                hints=(int(target.ships) + 1, int(target.ships) + enemy_ships + 1),
                max_turn=must_arrive_by,
            )
            if seeded is None:
                continue
            probe, rough_aim = seeded
            rough_turns = rough_aim[1]
            if rough_turns > must_arrive_by:
                continue

            plan = settle_plan(
                src, target, src_available, probe, world, planned_commitments,
                modes, policy, mission="capture",
                router=router,
            )
            if plan is None:
                continue
            angle, turns, _, need, send_pref = plan
            if turns > must_arrive_by:
                continue
            if not candidate_time_valid(target, turns, world, LATE_CAPTURE_BUFFER):
                continue

            # Score: value of denial + value of capture
            value = target_value(target, turns, "interdiction", world, modes, policy)
            # Add denial bonus: enemy fleet will arrive and bounce off our garrison
            denial = enemy_ships * 0.5 + target.production * (earliest_eta - turns)
            value += denial * DENIAL_BONUS_MULT * 8   # large denial reward

            if value <= 0:
                continue

            score = apply_score_modifiers(
                value / (send_pref + turns * ATTACK_COST_TURN_WEIGHT + 1.0),
                target, "interdiction", world,
            )
            option = ShotOption(
                score=score, src_id=src.id, target_id=target.id,
                angle=angle, turns=turns, needed=need, send_cap=send_pref,
                mission="capture", anchor_turn=must_arrive_by,
            )
            missions.append(Mission(kind="single", score=score, target_id=target.id, turns=turns, options=[option]))

    return missions


# ── NEW: Elimination surge missions ──────────────────────────────────────────
# When a specific enemy is weak, we build dedicated missions to wipe them out
# this turn rather than waiting for the normal priority queue.

def build_elimination_surge_missions(world, policy, planned_commitments, modes, router=None):
    """
    Target a dying opponent with maximum commitment from all available planets.
    Returns high-score missions to bump them to the top of the queue.
    """
    missions = []
    if not modes["is_elimination_surge"]:
        return missions

    # Find the weakest enemy
    weakest_opp  = None
    weakest_str  = 10 ** 9
    for opp_id in world.opp_planets:
        s = world.owner_strength.get(opp_id, 0)
        if s < weakest_str:
            weakest_str = s
            weakest_opp = opp_id

    if weakest_opp is None:
        return missions

    opp_planet_list = sorted(world.opp_planets[weakest_opp], key=lambda p: p.ships)
    # Attack the top-K weakest planets of this opponent
    for target in opp_planet_list[:ELIMINATION_SURGE_TOP_K]:
        for src in sorted(world.my_planets, key=lambda p: -policy["attack_budget"].get(p.id, 0)):
            src_available = policy["attack_budget"].get(src.id, 0)
            if src_available < ELIMINATION_SURGE_SHIPS:
                continue
            seeded = world.best_probe_aim(
                src.id, target.id, src_available,
                hints=(int(target.ships) + 1,),
            )
            if seeded is None:
                continue
            probe, rough_aim = seeded
            plan = settle_plan(
                src, target, src_available, probe, world, planned_commitments,
                modes, policy, mission="capture", router=router,
            )
            if plan is None:
                continue
            angle, turns, _, need, send_pref = plan
            if not candidate_time_valid(target, turns, world, LATE_CAPTURE_BUFFER):
                continue
            value = target_value(target, turns, "capture", world, modes, policy)
            if value <= 0:
                continue
            # Surge missions get a score inflator so they beat normal missions
            score = apply_score_modifiers(
                value / (send_pref + turns * ATTACK_COST_TURN_WEIGHT + 1.0),
                target, "capture", world,
            ) * 3.0    # triple score to ensure they fire

            option = ShotOption(
                score=score, src_id=src.id, target_id=target.id,
                angle=angle, turns=turns, needed=need, send_cap=send_pref,
                mission="capture",
            )
            missions.append(Mission(kind="single", score=score, target_id=target.id, turns=turns, options=[option]))
            break  # one source per target in the surge list

    return missions


# ============================================================
# Main planning function
# ============================================================

def plan_moves(world, deadline=None):
    def expired():
        return deadline is not None and time.perf_counter() > deadline

    def time_left():
        return (deadline - time.perf_counter()) if deadline is not None else 10 ** 9

    def allow_heavy_phase():
        return time_left() > HEAVY_PHASE_MIN_TIME and len(world.planets) <= HEAVY_ROUTE_PLANET_LIMIT

    def allow_optional_phase():
        return time_left() > OPTIONAL_PHASE_MIN_TIME

    modes  = build_modes(world)
    policy = build_policy_state(world, deadline=deadline)
    router = RouteOptimizer(world)
    planned_commitments    = defaultdict(list)
    source_options_by_target = defaultdict(list)
    missions = []
    moves    = []
    spent_total = defaultdict(int)

    def source_inventory_left(source_id):
        return world.source_inventory_left(source_id, spent_total)

    def source_attack_left(source_id):
        budget = policy["attack_budget"].get(source_id, 0)
        return max(0, budget - spent_total[source_id])

    def append_move(src_id, angle, ships):
        send = min(int(ships), source_inventory_left(src_id))
        if send < 1:
            return 0
        moves.append([src_id, float(angle), int(send)])
        spent_total[src_id] += send
        return send

    def finalize_moves():
        final_moves = []
        used_final  = defaultdict(int)
        for src_id, angle, ships in moves:
            source     = world.planet_by_id[src_id]
            max_allowed = int(source.ships) - used_final[src_id]
            send        = min(int(ships), max_allowed)
            if send >= 1:
                final_moves.append([src_id, float(angle), int(send)])
                used_final[src_id] += send
        return final_moves

    def compute_live_doomed():
        doomed = set()
        for planet in world.my_planets:
            status = world.hold_status(
                planet.id, planned_commitments=planned_commitments, horizon=DOOMED_EVAC_TURN_LIMIT
            )
            if (
                not status["holds_full"]
                and status["fall_turn"] is not None
                and status["fall_turn"] <= DOOMED_EVAC_TURN_LIMIT
                and source_inventory_left(planet.id) >= DOOMED_MIN_SHIPS
            ):
                doomed.add(planet.id)
        return doomed

    def time_filters_pass(target, turns, needed, src_cap):
        if not candidate_time_valid(target, turns, world, VERY_LATE_CAPTURE_BUFFER if world.is_very_late else LATE_CAPTURE_BUFFER):
            return False
        if opening_filter(target, turns, needed, src_cap, world, policy):
            return False
        return True

    # ── Phase 1: defence & recovery missions ─────────────────────────────────
    if allow_heavy_phase():
        missions.extend(build_reinforce_missions(world, policy, planned_commitments, modes, source_inventory_left))
    missions.extend(build_rescue_missions(world, policy, planned_commitments, modes, router=router))
    missions.extend(build_recapture_missions(world, policy, planned_commitments, modes, router=router))

    # ── Phase 2: elimination surge (NEW — highest priority offensive) ─────────
    if allow_heavy_phase():
        missions.extend(build_elimination_surge_missions(world, policy, planned_commitments, modes, router=router))

    # ── Phase 3: interdiction (NEW — race enemy to neutral planets) ───────────
    if allow_heavy_phase() and not expired():
        missions.extend(build_interdiction_missions(world, policy, planned_commitments, modes, router=router))

    # ── Phase 4: standard attack / snipe / swarm missions ────────────────────
    for src in world.my_planets:
        if expired():
            return finalize_moves()
        src_available = source_attack_left(src.id)
        if src_available <= 0:
            continue

        for target in world.planets:
            if expired():
                return finalize_moves()
            if target.id == src.id or target.owner == world.player:
                continue

            seeded = world.best_probe_aim(src.id, target.id, src_available, hints=(int(target.ships) + 1,))
            route  = None
            if seeded:
                _, rough_aim = seeded
                rough_turns  = rough_aim[1]
            else:
                route = router.find_2hop_route(src.id, target.id, src_available)
                if route:
                    rough_turns = route["total"]
                else:
                    continue

            if not candidate_time_valid(
                target, rough_turns, world,
                VERY_LATE_CAPTURE_BUFFER if world.is_very_late else LATE_CAPTURE_BUFFER,
            ):
                continue

            global_needed = world.min_ships_to_own_at(
                target.id, rough_turns, world.player,
                planned_commitments=planned_commitments,
            )
            if global_needed <= 0:
                continue
            if opening_filter(target, rough_turns, global_needed, src_available, world, policy):
                continue

            # Partial (swarm) option
            partial_send_cap = min(
                src_available,
                preferred_send(target, global_needed, rough_turns, src_available, world, modes, policy),
            )
            if partial_send_cap >= PARTIAL_SOURCE_MIN_SHIPS:
                partial_seeded = world.best_probe_aim(
                    src.id, target.id, partial_send_cap,
                    hints=(partial_send_cap, global_needed, int(target.ships) + 1),
                )
                if partial_seeded is not None:
                    _, partial_aim = partial_seeded
                    p_angle, p_turns = partial_aim[0], partial_aim[1]
                    if time_filters_pass(target, p_turns, global_needed, src_available):
                        partial_value = target_value(target, p_turns, "swarm", world, modes, policy)
                        if partial_value > 0:
                            p_score = apply_score_modifiers(
                                partial_value / (partial_send_cap + p_turns * ATTACK_COST_TURN_WEIGHT + 1.0),
                                target, "swarm", world,
                            )
                            source_options_by_target[target.id].append(
                                ShotOption(
                                    score=p_score, src_id=src.id, target_id=target.id,
                                    angle=p_angle, turns=p_turns, needed=global_needed,
                                    send_cap=partial_send_cap, mission="swarm",
                                )
                            )

            # Full-capture option
            if global_needed <= src_available:
                send_guess = preferred_send(
                    target, global_needed, rough_turns, src_available, world, modes, policy, src=src
                )
                plan = settle_plan(
                    src, target, src_available, send_guess, world, planned_commitments,
                    modes, policy, mission="capture", router=router,
                )
                if plan is not None:
                    angle, turns, _, needed, send_cap = plan
                    if time_filters_pass(target, turns, needed, src_available) and send_cap >= 1:
                        value = target_value(target, turns, "capture", world, modes, policy)
                        if value > 0:
                            score = apply_score_modifiers(
                                value / (send_cap + turns * ATTACK_COST_TURN_WEIGHT + 1.0),
                                target, "capture", world,
                            )
                            option = ShotOption(
                                score=score, src_id=src.id, target_id=target.id,
                                angle=angle, turns=turns, needed=needed, send_cap=send_cap,
                                mission="capture",
                            )
                            if send_cap >= needed:
                                missions.append(Mission(
                                    kind="single", score=score, target_id=target.id, turns=turns, options=[option]
                                ))

            # Snipe & shadow-snipe
            snipe = build_snipe_mission(src, target, src_available, world, planned_commitments, modes, policy, router=router)
            if snipe:
                missions.append(snipe)
            shadow = build_shadow_snipe_missions(src, target, src_available, world, planned_commitments, modes, policy, router=router)
            if shadow:
                missions.append(shadow)

    # ── Phase 5: multi-source swarms ─────────────────────────────────────────
    for target_id, options in source_options_by_target.items():
        if expired():
            return finalize_moves()
        if len(options) < 2:
            continue
        target      = world.planet_by_id[target_id]
        top_options = sorted(options, key=lambda o: -o.score)[:MULTI_SOURCE_TOP_K]

        for i in range(len(top_options)):
            for j in range(i + 1, len(top_options)):
                first, second = top_options[i], top_options[j]
                if first.src_id == second.src_id:
                    continue
                pair_tol = swarm_eta_tolerance((first, second), target, world)
                if abs(first.turns - second.turns) > pair_tol:
                    continue
                joint_turn = max(first.turns, second.turns)
                total_cap  = first.send_cap + second.send_cap
                need = world.min_ships_to_own_at(
                    target_id, joint_turn, world.player,
                    planned_commitments=planned_commitments, upper_bound=total_cap,
                )
                if need <= 0 or first.send_cap >= need or second.send_cap >= need or total_cap < need:
                    continue
                value = target_value(target, joint_turn, "swarm", world, modes, policy)
                if value <= 0:
                    continue
                pair_score = apply_score_modifiers(
                    value / (need + joint_turn * ATTACK_COST_TURN_WEIGHT + 1.0),
                    target, "swarm", world,
                ) * MULTI_SOURCE_PLAN_PENALTY
                missions.append(Mission(kind="swarm", score=pair_score, target_id=target_id, turns=joint_turn, options=[first, second]))

        if (
            THREE_SOURCE_SWARM_ENABLED
            and allow_heavy_phase()
            and target.owner not in (-1, world.player)
            and int(target.ships) >= THREE_SOURCE_MIN_TARGET_SHIPS
            and len(top_options) >= 3
        ):
            for i in range(len(top_options)):
                for j in range(i + 1, len(top_options)):
                    for k in range(j + 1, len(top_options)):
                        if expired():
                            return finalize_moves()
                        trio = [top_options[i], top_options[j], top_options[k]]
                        if len({o.src_id for o in trio}) < 3:
                            continue
                        trio_tol = swarm_eta_tolerance(tuple(trio), target, world)
                        turn_vals = [o.turns for o in trio]
                        if max(turn_vals) - min(turn_vals) > trio_tol:
                            continue
                        joint_turn = max(turn_vals)
                        total_cap  = sum(o.send_cap for o in trio)
                        need = world.min_ships_to_own_at(
                            target_id, joint_turn, world.player,
                            planned_commitments=planned_commitments, upper_bound=total_cap,
                        )
                        if need <= 0 or total_cap < need:
                            continue
                        if any(trio[a].send_cap + trio[b].send_cap >= need for a in range(3) for b in range(a+1, 3)):
                            continue
                        value = target_value(target, joint_turn, "swarm", world, modes, policy)
                        if value <= 0:
                            continue
                        trio_score = apply_score_modifiers(
                            value / (need + joint_turn * ATTACK_COST_TURN_WEIGHT + 1.0),
                            target, "swarm", world,
                        ) * THREE_SOURCE_PLAN_PENALTY
                        missions.append(Mission(kind="swarm", score=trio_score, target_id=target_id, turns=joint_turn, options=trio))

    # ── Phase 6: crash exploit ───────────────────────────────────────────────
    if allow_heavy_phase():
        missions.extend(build_crash_exploit_missions(world, policy, planned_commitments, modes, router=router))

    # ── Execute missions by score ─────────────────────────────────────────────
    missions.sort(key=lambda m: -m.score)

    for mission in missions:
        if expired():
            return finalize_moves()
        target = world.planet_by_id[mission.target_id]

        if mission.kind in ("single", "snipe", "rescue", "recapture", "reinforce", "crash_exploit", "shadow_snipe"):
            option = mission.options[0]
            src    = world.planet_by_id[option.src_id]
            if mission.kind == "reinforce":
                left = min(source_inventory_left(option.src_id), int(src.ships * REINFORCE_MAX_SOURCE_FRACTION))
            else:
                left = source_attack_left(option.src_id)
            if left <= 0:
                continue

            if mission.kind == "reinforce":
                plan = settle_reinforce_plan(
                    src, target, left, min(left, option.send_cap), world, planned_commitments,
                    option.anchor_turn, mission.turns,
                )
            elif mission.kind == "rescue":
                plan = settle_plan(
                    src, target, left, min(left, option.send_cap), world, planned_commitments,
                    modes, policy, mission="rescue",
                    eval_turn_fn=lambda _t, ht=mission.turns: ht,
                    anchor_turn=option.anchor_turn, router=router,
                )
            elif mission.kind == "snipe":
                plan = settle_plan(
                    src, target, left, min(left, option.send_cap), world, planned_commitments,
                    modes, policy, mission="snipe",
                    eval_turn_fn=lambda t, e=option.anchor_turn: max(t, e),
                    anchor_turn=option.anchor_turn, router=router,
                )
            elif mission.kind == "shadow_snipe":
                plan = settle_plan(
                    src, target, left, min(left, option.send_cap), world, planned_commitments,
                    modes, policy, mission="snipe",
                    anchor_turn=option.anchor_turn, anchor_tolerance=1, router=router,
                )
            elif mission.kind == "crash_exploit":
                plan = settle_plan(
                    src, target, left, min(left, option.send_cap), world, planned_commitments,
                    modes, policy, mission="crash_exploit",
                    eval_turn_fn=lambda t, d=option.anchor_turn: max(t, d),
                    anchor_turn=option.anchor_turn, anchor_tolerance=CRASH_EXPLOIT_ETA_WINDOW,
                    router=router,
                )
            else:
                plan = settle_plan(
                    src, target, left, min(left, option.send_cap), world, planned_commitments,
                    modes, policy, mission="capture", router=router,
                )

            if plan is None:
                continue
            angle, turns, _, need, send = plan
            if send < need or need > left:
                continue
            sent = append_move(option.src_id, angle, send)
            if sent < need:
                continue
            planned_commitments[target.id].append((turns, world.player, int(sent)))
            continue

        # Multi-source swarm
        limits  = [min(source_attack_left(o.src_id), o.send_cap) for o in mission.options]
        if min(limits) <= 0:
            continue
        missing = world.min_ships_to_own_at(
            target.id, mission.turns, world.player,
            planned_commitments=planned_commitments, upper_bound=sum(limits),
        )
        if missing <= 0 or sum(limits) < missing:
            continue
        ordered   = sorted(zip(mission.options, limits), key=lambda x: (x[0].turns, -x[1], x[0].src_id))
        remaining = missing
        sends     = {}
        for idx, (option, limit) in enumerate(ordered):
            remaining_other = sum(ol for _, ol in ordered[idx + 1:])
            send = min(limit, max(0, remaining - remaining_other))
            sends[option.src_id] = send
            remaining -= send
        if remaining > 0:
            continue

        reaimed = []
        for option, _ in ordered:
            send = sends.get(option.src_id, 0)
            if send <= 0:
                continue
            src      = world.planet_by_id[option.src_id]
            fixed    = world.plan_shot(src.id, target.id, send)
            if fixed is None:
                reaimed = []
                break
            angle, turns, _, _ = fixed
            reaimed.append((option.src_id, angle, turns, send))
        if not reaimed:
            continue

        turns_only = [item[2] for item in reaimed]
        eta_tol    = swarm_eta_tolerance(mission.options, target, world)
        if max(turns_only) - min(turns_only) > eta_tol:
            continue

        actual_joint = max(turns_only)
        owner_after, _ = world.projected_state(
            target.id, actual_joint,
            planned_commitments=planned_commitments,
            extra_arrivals=[(t, world.player, s) for _, _, t, s in reaimed],
        )
        if owner_after != world.player:
            continue

        committed = []
        for src_id, angle, turns, send in reaimed:
            actual = append_move(src_id, angle, send)
            if actual > 0:
                committed.append((turns, world.player, int(actual)))
        if sum(c[2] for c in committed) < missing:
            continue
        planned_commitments[target.id].extend(committed)

    # ── Phase 7: follow-up pass with leftover budget ──────────────────────────
    if not world.is_very_late and allow_optional_phase():
        for src in world.my_planets:
            if expired():
                return finalize_moves()
            src_left = source_attack_left(src.id)
            if src_left < FOLLOWUP_MIN_SHIPS:
                continue

            best = None
            for target in world.planets:
                if expired():
                    return finalize_moves()
                if target.id == src.id or target.owner == world.player:
                    continue
                if target.id in world.comet_ids and target.production <= LOW_VALUE_COMET_PRODUCTION:
                    continue
                seeded = world.best_probe_aim(src.id, target.id, src_left, hints=(int(target.ships) + 1,))
                if seeded is None:
                    continue
                _, rough_aim = seeded
                est_turns    = rough_aim[1]
                if world.is_late and est_turns > world.remaining_steps - LATE_CAPTURE_BUFFER:
                    continue
                rough_needed = world.min_ships_to_own_at(
                    target.id, est_turns, world.player,
                    planned_commitments=planned_commitments, upper_bound=src_left,
                )
                if rough_needed <= 0 or rough_needed > src_left:
                    continue
                if opening_filter(target, est_turns, rough_needed, src_left, world, policy):
                    continue
                send = preferred_send(target, rough_needed, est_turns, src_left, world, modes, policy, src=src)
                if send < rough_needed:
                    continue
                plan = settle_plan(
                    src, target, src_left, send, world, planned_commitments,
                    modes, policy, mission="capture", router=router,
                )
                if plan is None:
                    continue
                _, turns, _, need, final_send = plan
                if world.is_late and turns > world.remaining_steps - LATE_CAPTURE_BUFFER:
                    continue
                if final_send < need:
                    continue
                value = target_value(target, turns, "capture", world, modes, policy)
                if value <= 0:
                    continue
                score = apply_score_modifiers(
                    value / (final_send + turns * ATTACK_COST_TURN_WEIGHT + 1.0),
                    target, "capture", world,
                )
                if best is None or score > best[0]:
                    best = (score, target, plan)

            if best is None:
                continue
            _, target, plan = best
            angle, turns, _, need, send = plan
            src_left = source_attack_left(src.id)
            if need > src_left:
                continue
            plan = settle_plan(
                src, target, src_left, min(src_left, send), world, planned_commitments,
                modes, policy, mission="capture", router=router,
            )
            if plan is None:
                continue
            angle, turns, _, need, send = plan
            if send < need:
                continue
            actual = append_move(src.id, angle, send)
            if actual < need:
                continue
            planned_commitments[target.id].append((turns, world.player, int(actual)))

    # ── Phase 8: doomed planet evacuation ────────────────────────────────────
    if expired():
        return finalize_moves()
    live_doomed = compute_live_doomed()
    if live_doomed:
        frontier_targets = world.enemy_planets or world.static_neutral_planets or world.neutral_planets
        if frontier_targets:
            frontier_distance = {
                p.id: nearest_distance_to_set(p.x, p.y, frontier_targets)
                for p in world.my_planets
            }
        else:
            frontier_distance = {p.id: 10 ** 9 for p in world.my_planets}

        for planet in world.my_planets:
            if expired():
                return finalize_moves()
            if planet.id not in live_doomed:
                continue
            available_now = source_inventory_left(planet.id)
            if available_now < policy["reserve"].get(planet.id, 0):
                continue

            best_capture = None
            for target in world.planets:
                if expired():
                    return finalize_moves()
                if target.id == planet.id or target.owner == world.player:
                    continue
                seeded = world.best_probe_aim(
                    planet.id, target.id, available_now,
                    hints=(available_now, int(target.ships) + 1),
                )
                if seeded is None:
                    continue
                _, probe_aim   = seeded
                probe_turns    = probe_aim[1]
                if probe_turns > world.remaining_steps - 2:
                    continue
                need = world.min_ships_to_own_at(
                    target.id, probe_turns, world.player,
                    planned_commitments=planned_commitments, upper_bound=available_now,
                )
                if need <= 0 or need > available_now:
                    continue
                plan = settle_plan(
                    planet, target, available_now,
                    min(available_now, max(need, int(target.ships) + 1)),
                    world, planned_commitments, modes, policy, mission="capture",
                )
                if plan is None:
                    continue
                angle, turns, _, plan_need, send = plan
                if send < plan_need:
                    continue
                score = target_value(target, turns, "capture", world, modes, policy) / (send + turns + 1.0)
                if target.owner not in (-1, world.player):
                    score *= 1.05
                if best_capture is None or score > best_capture[0]:
                    best_capture = (score, target.id, angle, turns, send)

            if best_capture is not None:
                _, target_id, angle, turns, need = best_capture
                actual = append_move(planet.id, angle, need)
                if actual >= 1:
                    planned_commitments[target_id].append((turns, world.player, int(actual)))
                continue

            safe_allies = [a for a in world.my_planets if a.id != planet.id and a.id not in live_doomed]
            if not safe_allies:
                continue
            retreat = min(safe_allies, key=lambda a: (frontier_distance.get(a.id, 10**9), planet_distance(planet, a)))
            aim     = world.plan_shot(planet.id, retreat.id, available_now)
            if aim is None:
                continue
            append_move(planet.id, aim[0], available_now)

    # ── Phase 9: rear-planet forwarding ──────────────────────────────────────
    if (
        (world.enemy_planets or world.neutral_planets)
        and len(world.my_planets) > 1
        and not world.is_late
        and allow_optional_phase()
    ):
        live_doomed = compute_live_doomed()
        frontier_targets = world.enemy_planets or world.static_neutral_planets or world.neutral_planets
        frontier_distance = {
            p.id: nearest_distance_to_set(p.x, p.y, frontier_targets)
            for p in world.my_planets
        }
        safe_fronts = [p for p in world.my_planets if p.id not in live_doomed]
        if safe_fronts:
            front_anchor = min(safe_fronts, key=lambda p: frontier_distance[p.id])
            send_ratio   = REAR_SEND_RATIO_FOUR_PLAYER if world.is_four_player else REAR_SEND_RATIO_TWO_PLAYER
            if modes["is_finishing"] or modes["is_avalanche"]:
                send_ratio = max(send_ratio, REAR_SEND_RATIO_FOUR_PLAYER)

            for rear in sorted(world.my_planets, key=lambda p: -frontier_distance[p.id]):
                if expired():
                    return finalize_moves()
                if rear.id == front_anchor.id or rear.id in live_doomed:
                    continue
                if source_attack_left(rear.id) < REAR_SOURCE_MIN_SHIPS:
                    continue
                if frontier_distance[rear.id] < frontier_distance[front_anchor.id] * REAR_DISTANCE_RATIO:
                    continue
                stage_candidates = [
                    p for p in safe_fronts
                    if p.id != rear.id
                    and frontier_distance[p.id] < frontier_distance[rear.id] * REAR_STAGE_PROGRESS
                ]
                if stage_candidates:
                    front = min(stage_candidates, key=lambda p: planet_distance(rear, p))
                else:
                    objective = min(frontier_targets, key=lambda t: planet_distance(rear, t))
                    remaining_fronts = [p for p in safe_fronts if p.id != rear.id]
                    if not remaining_fronts:
                        continue
                    front = min(remaining_fronts, key=lambda p: planet_distance(p, objective))
                if front.id == rear.id:
                    continue
                send = int(source_attack_left(rear.id) * send_ratio)
                if send < REAR_SEND_MIN_SHIPS:
                    continue
                aim = world.plan_shot(rear.id, front.id, send)
                if aim is None:
                    continue
                _, turns, _, _ = aim
                if turns > REAR_MAX_TRAVEL_TURNS:
                    continue
                append_move(rear.id, aim[0], send)

    return finalize_moves()


# ============================================================
# Agent entry point
# ============================================================

def _read(obs, key, default=None):
    if isinstance(obs, dict):
        return obs.get(key, default)
    return getattr(obs, key, default)


def build_world(obs):
    player      = _read(obs, "player", 0)
    step        = _read(obs, "step", 0) or 0
    raw_planets = _read(obs, "planets", []) or []
    raw_fleets  = _read(obs, "fleets",  []) or []
    ang_vel     = _read(obs, "angular_velocity", 0.0) or 0.0
    raw_init    = _read(obs, "initial_planets", []) or []
    comets      = _read(obs, "comets", []) or []
    comet_ids   = set(_read(obs, "comet_planet_ids", []) or [])

    planets          = [Planet(*p) for p in raw_planets]
    fleets           = [Fleet(*f)  for f in raw_fleets]
    initial_planets  = [Planet(*p) for p in raw_init]
    initial_by_id    = {p.id: p for p in initial_planets}

    return WorldModel(
        player=player, step=step, planets=planets, fleets=fleets,
        initial_by_id=initial_by_id, ang_vel=ang_vel, comets=comets, comet_ids=comet_ids,
    )


def agent(obs, config=None):
    start_time  = time.perf_counter()
    world       = build_world(obs)
    if not world.my_planets:
        return []
    act_timeout = _read(config, "actTimeout", 1.0) if config is not None else 1.0
    soft_budget = min(SOFT_ACT_DEADLINE, max(0.55, act_timeout * 0.82))
    deadline    = start_time + soft_budget
    return plan_moves(world, deadline=deadline)


__all__ = ["agent", "build_world"]
```

---

# 3. Core Design Philosophy

The agent treats Orbit Wars as a compounding economy problem.

Earlier production advantages create exponential long-term gains.

Example:

```text
Capture Planet Early
    ↓
More Production
    ↓
More Ships
    ↓
More Captures
    ↓
Map Control
    ↓
Economic Avalanche
```

Because of this, the bot aggressively optimizes:

- Tempo
- Production timing
- Expansion efficiency
- Resource conservation

---

# 4. Major Improvements Over Baseline Agents

| System | Purpose |
|---|---|
| NPV Target Scoring | Prioritizes early economic gain |
| Threat Reserve Maps | Prevents overextension |
| Speed Bracket Exploitation | Uses logarithmic speed thresholds |
| Enemy Interdiction | Beats enemies to valuable targets |
| Elimination Surge | Finishes weak opponents quickly |
| Avalanche Mode | Converts production lead into domination |
| Predictive Orbit Intercepts | Hits moving planets accurately |
| Multi-Source Swarms | Coordinates synchronized attacks |

---

# 5. Imports

```python
import math
import time

from collections import defaultdict, namedtuple
from dataclasses import dataclass, field
```

---

# 6. Configuration

This section defines all strategic tuning constants.

---

## 6.1 Map Parameters

```python
BOARD       = 100.0

CENTER_X    = 50.0
CENTER_Y    = 50.0

SUN_R       = 10.0
MAX_SPEED   = 6.0

ROTATION_LIMIT = 50.0
TOTAL_STEPS    = 500
```

---

## 6.2 Strategic Modes

```python
EARLY_TURN_LIMIT = 65
OPENING_TURN_LIMIT = 138

LATE_REMAINING_TURNS = 60
VERY_LATE_REMAINING_TURNS = 25
```

### Game Phases

| Phase | Strategy |
|---|---|
| Opening | Aggressive expansion |
| Midgame | Controlled pressure |
| Avalanche | Full domination |
| Elimination | Finish weak opponents |
| Endgame | Immediate-value optimization |

---

## 6.3 Strategic Multipliers

```python
HOSTILE_TARGET_VALUE_MULT = 2.11
INTERDICTION_VALUE_MULT   = 1.93
ELIMINATION_VALUE_MULT    = 1.76
```

Higher values increase targeting priority.

---

## 6.4 NPV Compounding Logic

```python
NPV_COMPOUNDING_RATE = 0.177
EARLY_RUSH_MULT      = 0.49
```

Earlier captures gain extra value because they generate future production sooner.

---

# 7. Shared Data Structures

---

## 7.1 Planet

```python
Planet = namedtuple(
    "Planet",
    [
        "id",
        "owner",
        "x",
        "y",
        "radius",
        "ships",
        "production"
    ]
)
```

---

## 7.2 Fleet

```python
Fleet = namedtuple(
    "Fleet",
    [
        "id",
        "owner",
        "x",
        "y",
        "angle",
        "from_planet_id",
        "ships"
    ]
)
```

---

## 7.3 ShotOption

Represents a possible attack plan.

```python
@dataclass(frozen=True)
class ShotOption:
    score: float
    src_id: int
    target_id: int
    angle: float
    turns: int
    needed: int
    send_cap: int
```

---

# 8. Physics Engine

The physics system predicts:

- Fleet travel
- Planet rotation
- Orbital interception
- Collision geometry
- Safe routing

---

## 8.1 Fleet Speed

Fleet speed scales logarithmically.

```python
def fleet_speed(ships):
    if ships <= 1:
        return 1.0

    ratio = math.log(ships) / math.log(1000.0)

    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)
```

---

## 8.2 Speed Brackets

Small ship-count increases can reduce ETA by an entire turn.

```text
Ships Sent
    ↓
Higher Speed Tier
    ↓
Lower ETA
    ↓
Earlier Capture
```

The bot explicitly exploits these thresholds.

---

## 8.3 Safe Pathing

The agent avoids trajectories through the sun.

```python
segment_hits_sun(...)
```

This prevents:

- suicidal paths
- invalid trajectories
- blocked movement

---

## 8.4 Predictive Interception

Moving planets require iterative interception prediction.

Pipeline:

```text
Predict Position
    ↓
Estimate Arrival
    ↓
Refine Future Position
    ↓
Recompute ETA
    ↓
Converge
```

Implemented using:

```python
aim_with_prediction(...)
```

---

# 9. World Simulation System

The bot internally simulates future game states.

---

## 9.1 Arrival Ledger

All incoming fleets are indexed as:

```text
planet_id → future arrivals
```

This enables future combat prediction.

---

## 9.2 Timeline Simulation

Each planet simulates:

- future ownership
- future ship counts
- collapse timing
- required reserve strength

Implemented using:

```python
simulate_planet_timeline(...)
```

---

## 9.3 Threat Maps

Instead of fixed reserve ratios, the bot computes:

```text
minimum ships needed for survival
```

Benefits:

- safer expansion
- stronger defense
- fewer collapses
- improved multi-front survival

---

# 10. Strategic Systems

---

# 10.1 Opening Expansion

Opening logic aggressively captures:

```text
high production / low ETA neutrals
```

using:

```python
(production / turns)^1.5
```

---

# 10.2 Enemy Interdiction

The bot scans enemy fleets heading toward neutral planets.

If an enemy attempts:

```text
Enemy → Neutral Planet
```

the agent may:

- arrive first
- intercept after combat
- steal the planet efficiently

This creates large economic swings.

---

# 10.3 Elimination Surge

When an opponent becomes weak:

```python
enemy_strength < ELIMINATION_STRENGTH_RATIO * my_strength
```

the agent enters:

```text
SURGE MODE
```

All available fleets coordinate to eliminate the opponent rapidly.

---

# 10.4 Avalanche Mode

If production advantage becomes overwhelming:

```python
my_prod / enemy_prod > AVALANCHE_PROD_RATIO
```

attack allocation increases dramatically.

```python
AVALANCHE_ATTACK_FRACTION = 0.85
```

The bot converts economic advantage into direct domination.

---

# 10.5 Multi-Source Swarms

Targets can be attacked using:

- dual-source attacks
- triple-source swarms
- synchronized ETAs

This enables captures impossible with isolated fleets.

---

# 11. Tactical Scoring System

Every target receives a composite score.

---

## 11.1 Economic Value

```text
Production gained
```

---

## 11.2 Travel Cost

```text
Turns required
```

---

## 11.3 Strategic Positioning

Indirect influence considers:

- nearby enemies
- nearby neutrals
- nearby allies

---

## 11.4 Tempo Bonus

Earlier captures receive:

```text
future production snowball value
```

---

# 12. Defensive Intelligence

Defense is proactive rather than reactive.

The bot predicts:

- future enemy stacks
- reinforcement timings
- multi-wave attacks
- collapse windows

before committing aggression.

---

# 13. Reinforcement Logic

Rear planets reinforce frontlines when:

- local threat is low
- ETA is acceptable
- future survival remains stable

This prevents idle ship accumulation.

---

# 14. Late Game Logic

During endgame:

- immediate ship value increases
- long-term economy matters less
- direct captures become priority

The strategy shifts toward:

```text
conversion efficiency
```

instead of growth.

---

# 15. Execution Pipeline

```text
Observe Game State
        ↓
Build World Model
        ↓
Simulate Future Timelines
        ↓
Compute Threat Reserves
        ↓
Generate Candidate Attacks
        ↓
Score Targets
        ↓
Apply Strategic Modes
        ↓
Validate Survivability
        ↓
Launch Fleets
```

---

# 16. Future ML Extensions

The architecture is fully compatible with ML systems.

Potential upgrades:

| ML System | Purpose |
|---|---|
| Shot Validator | Reject low-success attacks |
| PPO Policy Layer | Strategic mode selection |
| Learned Target Scoring | Replace heuristics |
| Fleet Value Estimator | Learned combat efficiency |
| Replay Distillation | Supervised improvement |

---

# 17. Submission Structure

Recommended competition structure:

```text
submission.py
weights.npz
decode_weights.py
```

Future additions:

```text
validator.py
trainer.ipynb
selfplay.py
```

---

# 18. Performance Characteristics

## Strengths

- Strong macro economy
- Excellent expansion tempo
- Efficient ship conservation
- High-quality orbital prediction
- Good multi-player scaling

## Weaknesses

- Computationally expensive
- Large tuning surface
- Timing-sensitive heuristics
- Complex debugging

---

# 19. Conclusion

This agent combines:

- predictive simulation
- economic optimization
- orbital interception
- coordinated swarming
- adaptive strategic modes

into a highly competitive Orbit Wars framework.

The notebook structure also prepares the project for:

- ML augmentation
- tournament experimentation
- replay analytics
- Kaggle competition workflows